# SupplementRx AI — Data Audit and Preprocessing

**Phase 1: audit only.** Inspect all six sources without changing any raw file. NatMed Pro and Reddit already contain upstream normalization, passages and graph exports; the audit examines those artifacts rather than assuming they are unprocessed browser captures.

Use a fresh Python kernel and run cells in order from the project root or `notebooks/`. No API calls, credentials, database connection or new package installation are required. The notebook keeps diagnostics in memory and prints counts and record identifiers, not source text or usernames.

**Phase 2: DSLD label inventory only.** Section 8 implements the first preprocessing step: market-status flags and repeated-name review groups, while preserving every raw label. It writes versioned outputs under `data/processed/dsld/`. Sections 1–6 remain read-only audits. Ingredient transformations, cross-source mappings, Reddit preprocessing and clinical claim processing remain pending.

Audit checks cover file integrity, record structure, duplicate IDs/text, label quantities and forms, candidate ambiguity, source and passage bindings, citation gaps, Reddit context, and evidence/review flags. Passing mechanical checks does not validate clinical truth or certify source capture completeness.


In [1]:
import json
import hashlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data/raw'
# Explicit snapshots: never silently substitute a newer terminology run.
RUNS = {
    'pubchem': '20261028T035149634190Z',
    'rxnorm': '20261028T060819025094Z',
    'mesh': '20261028T062833838481Z',
}
SOURCE_DIRS = {name: RAW_DIR / name for name in ('dsld','natmedpro','reddit')}
SOURCE_DIRS.update({name: RAW_DIR/name/'runs'/run for name,run in RUNS.items()})
audit_started_at = datetime.now(timezone.utc).isoformat()
findings = []
summary = {}
source_inventory = {}

def sha_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def raw_snapshot():
    return {str(p.relative_to(RAW_DIR)): sha_file(p)
            for p in sorted(RAW_DIR.rglob('*')) if p.is_file() and p.name != '.DS_Store'}

def finding(source, severity, check, count, examples=None):
    if count:
        findings.append({'source': source, 'severity': severity, 'check': check,
                         'count': int(count), 'examples': list(examples or [])[:5]})

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl(path):
    rows = []
    with path.open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row, dict):
                raise ValueError(f'{path}: line {line_number} is not an object')
            rows.append(row)
    return rows

def check_records(source, rows, required):
    ids = Counter(str(row.get('id')) for row in rows)
    duplicate_ids = [key for key,n in ids.items() if n > 1]
    missing = [str(row.get('id',i)) for i,row in enumerate(rows)
               if any(row.get(field) is None or row.get(field) == '' for field in required)]
    finding(source,'error','duplicate_record_ids',len(duplicate_ids),duplicate_ids)
    finding(source,'error','missing_required_fields',len(missing),missing)

raw_before = raw_snapshot()
print('Raw files fingerprinted:', len(raw_before))
print('No raw writes or network requests are performed.')


Raw files fingerprinted: 488
No raw writes or network requests are performed.


## 1. Verify source manifests and provenance
Check the current source files against their saved sizes and SHA-256 hashes. Source and run identifiers are recorded in the audit inventory. A parsing or missing-file error stops the notebook; checksum failures are reported as errors and block readiness for preprocessing.


In [2]:
for source, directory in SOURCE_DIRS.items():
    manifest_path = directory / 'collection_manifest.json'
    manifest = read_json(manifest_path)
    source_inventory[source] = {'directory': str(directory.relative_to(PROJECT_ROOT)),
        'manifest_sha256': sha_file(manifest_path),
        'source_commit': manifest.get('source_commit'), 'run': RUNS.get(source)}
    if source in RUNS:
        path = directory / manifest['collection_file']
        valid = path.stat().st_size == manifest['bytes'] and sha_file(path) == manifest['sha256']
        finding(source,'error','collection_integrity_failure',not valid)
    elif source in ('natmedpro','reddit'):
        for name, metadata in manifest['files'].items():
            path = directory / name
            valid = path.stat().st_size == metadata['bytes'] and sha_file(path) == metadata['sha256']
            finding(source,'error','file_integrity_failure',not valid,[name])
    else:
        combined = read_json(directory/'collection.json')
        checks = combined['source_file_sha256']
        for name, expected in checks.items():
            path = directory/'labels'/name if name.startswith('label_') else directory/name
            finding(source,'error','file_integrity_failure',sha_file(path)!=expected,[name])
        finding(source,'error','combined_manifest_mismatch',combined['collection_manifest'] != manifest)
    print(source, ': manifest checked')


dsld : manifest checked
natmedpro : manifest checked
reddit : manifest checked
pubchem : manifest checked
rxnorm : manifest checked
mesh : manifest checked


## 2. Audit DSLD products and ingredient structure
Count nested rows and forms without flattening or converting quantities. Off-market labels, repeated product names and missing quantities are review items, not automatic deletion rules. A product/brand name alone is not a reliable deduplication key.


In [3]:
dsld = read_json(SOURCE_DIRS['dsld']/'collection.json')
labels = dsld['labels']
check_records('dsld', labels, ['id','fullName','brandName'])
label_files = sorted((SOURCE_DIRS['dsld']/'labels').glob('label_*.json'))
individual = {str(row['id']):row for row in map(read_json,label_files)}
combined_labels = {str(row['id']):row for row in labels}
finding('dsld','error','combined_and_individual_labels_differ',individual != combined_labels)
finding('dsld','error','combined_record_count_mismatch',dsld['record_count'] != len(labels))

def ingredient_rows(rows):
    for row in rows or []:
        yield row
        yield from ingredient_rows(row.get('nestedRows'))

rows = [row for label in labels for row in ingredient_rows(label.get('ingredientRows'))]
name_groups = Counter((label.get('brandName'),label.get('fullName')) for label in labels)
repeat_names = sum(n>1 for n in name_groups.values())
off_market = sum(label.get('offMarket') == 1 for label in labels)
missing_quantity = sum(not row.get('quantity') for row in rows)
units = Counter(q.get('unit','<missing>') for row in rows for q in row.get('quantity') or [])
finding('dsld','review','off_market_labels',off_market)
finding('dsld','review','repeated_brand_product_name_groups',repeat_names)
finding('dsld','review','ingredient_rows_without_quantity',missing_quantity)
summary['dsld'] = {'labels':len(labels),'top_level_ingredient_rows':sum(len(l.get('ingredientRows') or []) for l in labels),
    'all_ingredient_rows':len(rows),'rows_with_nested_ingredients':sum(bool(r.get('nestedRows')) for r in rows),
    'form_entries':sum(len(r.get('forms') or []) for r in rows),
    'off_market_labels':off_market,'quantity_units':dict(units)}
print(json.dumps(summary['dsld'],indent=2))


{
  "labels": 400,
  "top_level_ingredient_rows": 1553,
  "all_ingredient_rows": 2339,
  "rows_with_nested_ingredients": 242,
  "form_entries": 1052,
  "off_market_labels": 231,
  "quantity_units": {
    "mg": 1060,
    "NP": 705,
    "mcg": 204,
    "IU": 56,
    "{Calories}": 26,
    "g": 8,
    "ug": 1,
    "Gram(s)": 213,
    "DU": 3,
    "HUT": 3,
    "FCCFIP": 3,
    "CU": 3,
    "INVU": 3,
    "ALU": 3,
    "DP": 6,
    "GaIU": 3,
    "Endo-PGU": 3,
    "XU": 3,
    "HCU": 3,
    "BGU": 3,
    "Kcal": 1,
    "Calorie(s)": 62,
    "grams": 2,
    "mL": 1,
    "mcg DFE": 12,
    "mcg RAE": 1,
    "mg NE": 1
  }
}


## 3. Audit PubChem, RxNorm and MeSH candidates
Measure ambiguity and no-match coverage separately from failures. Candidate returns are not accepted mappings. These audits inspect collected data only and do not retry APIs, infer missing IDs or merge entities.


In [4]:
for source in RUNS:
    bundle = read_json(SOURCE_DIRS[source]/'collection.json')
    queries = bundle['search_results']
    statuses = Counter(result['status'] for result in queries.values())
    finding(source,'error','failed_queries',statuses.get('error',0))
    finding(source,'review','no_match_queries',statuses.get('not_found',0))
    if source == 'pubchem':
        records = bundle['records']
        ambiguous = [k for k,r in queries.items() if len(r['cids'])>1]
        unresolved = [k for k,r in queries.items() if any(str(cid) not in records for cid in r['cids'])]
        bad = [cid for cid,r in records.items() if r.get('cid') != int(cid)
               or r.get('properties',{}).get('CID') != int(cid)
               or r.get('synonyms',{}).get('CID') != int(cid)]
        missing_name = [cid for cid,r in records.items() if not r.get('properties',{}).get('IUPACName')]
        finding(source,'review','missing_iupac_name',len(missing_name),missing_name)
    elif source == 'rxnorm':
        records = bundle['records']
        ambiguous = [k for k,r in queries.items() if len(r['rxcuis'])>1]
        unresolved = [k for k,r in queries.items() if any(rid not in records for rid in r['rxcuis'])]
        bad = [rid for rid,r in records.items() if r.get('properties',{}).get('response',{}).get('properties',{}).get('rxcui') != rid]
        normalized = [k for k,r in queries.items() if r.get('match_method')=='normalized']
        finding(source,'review','normalized_match_queries',len(normalized),normalized)
    else:
        index = bundle['candidate_record_index']
        records = {row['record']['value'] for batch in bundle['record_subgraph_responses']
                   for page in batch['pages'] if page['status']=='ok'
                   for row in page['response']['results']['bindings']
                   if row['subject']['value']==row['record']['value']}
        ambiguous = [k for k,v in index.items() if len(v)>1]
        unresolved = [k for k,v in index.items() if any(uri not in records for uri in v)]
        bad = bundle['report']['incomplete_record_uris']
        finding(source,'error','unresolved_entry_terms',len(bundle['report']['unresolved_term_uris']))
        finding(source,'error','incomplete_resolution_batches',bundle['report']['incomplete_resolution_batches'])
        finding(source,'error','possibly_truncated_queries',len(bundle['report']['possibly_truncated_queries']))
    finding(source,'review','ambiguous_candidate_queries',len(ambiguous),ambiguous)
    finding(source,'error','candidate_references_missing_record',len(unresolved),unresolved)
    finding(source,'error','invalid_or_incomplete_records',len(bad),bad)
    summary[source] = {'query_groups':len(queries),'statuses':dict(statuses),
                       'candidate_records':len(records),'ambiguous_queries':len(ambiguous)}
    print(source,json.dumps(summary[source]))
    del bundle


pubchem {"query_groups": 739, "statuses": {"ok": 228, "not_found": 511}, "candidate_records": 239, "ambiguous_queries": 7}
rxnorm {"query_groups": 739, "statuses": {"matched": 313, "not_found": 426}, "candidate_records": 287, "ambiguous_queries": 31}
mesh {"query_groups": 739, "statuses": {"not_found": 469, "matched": 270}, "candidate_records": 247, "ambiguous_queries": 6}


## 4. Audit NatMed Pro and Reddit records and passages
Check source roles, required fields, exact duplicate text, source hashes and section/word-window bindings. Duplicate text is counted for review; nothing is removed. Reddit missing context and inherited topics remain explicit. NatMed review flags and bibliography gaps remain unchanged.


In [5]:
source_rows = {}
for source in ('natmedpro','reddit'):
    directory = SOURCE_DIRS[source]
    rows = read_jsonl(directory/'records.jsonl')
    source_rows[source] = rows
    check_records(source,rows,['id','source','source_url','text','content_hash'])
    expected_source = 'natmed' if source=='natmedpro' else 'reddit'
    expected_role = 'clinical_reference' if source=='natmedpro' else 'user_report'
    bad_role = [r['id'] for r in rows if r.get('source')!=expected_source or r.get('evidence_role')!=expected_role]
    bad_hash = [r['id'] for r in rows if hashlib.sha256(r['text'].encode('utf-8')).hexdigest()!=r['content_hash']]
    finding(source,'error','unexpected_source_or_evidence_role',len(bad_role),bad_role)
    finding(source,'error','source_text_hash_mismatch',len(bad_hash),bad_hash)
    text_counts = Counter(r['text'] for r in rows)
    duplicates = [n for n in text_counts.values() if n>1]
    finding(source,'review','exact_duplicate_text_groups',len(duplicates))
    by_id = {r['id']:r for r in rows}
    passages = read_jsonl(directory/'upstream_derived/passages.jsonl')
    check_records(source+'_passages',passages,['id','record_id','text'])
    broken = []
    for passage in passages:
        try:
            record = by_id[passage['record_id']]
            sections = record.get('sections') or [{'heading':'discussion','text':record['text']}]
            section = sections[passage['section_index']]
            text = ' '.join(section['text'].split()[passage['word_start']:passage['word_start']+passage['word_count']])
            valid = (passage['text']==text and passage['section']==section['heading']
                and passage['source_content_hash']==record['content_hash']
                and passage['source']==expected_source and passage['evidence_role']==expected_role)
        except (KeyError,IndexError,TypeError):
            valid = False
        if not valid: broken.append(passage.get('id'))
    finding(source,'error','invalid_passage_source_binding',len(broken),broken)
    summary[source] = {'records':len(rows),'passages':len(passages),
        'record_types':dict(Counter(r['record_type'] for r in rows)),
        'duplicate_text_groups':len(duplicates),'records_in_duplicate_text_groups':sum(duplicates)}
    if source=='natmedpro':
        unreviewed = [r['id'] for r in rows if r.get('requires_clinical_review') or not (r.get('capture_quality') or {}).get('complete')]
        missing_refs = sum((r.get('capture_quality',{}).get('bibliography_coverage') or {}).get('inline_citation_refs_missing_from_bibliography',0) for r in rows)
        finding(source,'review','monographs_requiring_source_or_clinical_review',len(unreviewed),unreviewed)
        finding(source,'review','inline_reference_numbers_missing_from_bibliographies',missing_refs)
        summary[source]['source_titles'] = [r['title'] for r in rows]
        summary[source]['inline_reference_gaps'] = missing_refs
    else:
        native_ids = {r.get('native_id') for r in rows}
        saved_posts = {r.get('native_id') for r in rows if r['record_type']=='post'}
        missing_threads = [r['id'] for r in rows if r['record_type']=='comment' and r.get('thread_id') not in saved_posts]
        missing_parents = [r['id'] for r in rows if r['record_type']=='comment' and r.get('parent_id') and r['parent_id'] not in native_ids]
        finding(source,'review','comments_without_saved_thread_post',len(missing_threads),missing_threads)
        finding(source,'review','comments_without_saved_direct_parent',len(missing_parents),missing_parents)
        summary[source]['mention_scope'] = dict(Counter(r.get('mention_scope','<missing>') for r in rows))
        summary[source]['subreddits'] = dict(Counter(r.get('subreddit','<missing>') for r in rows))
        summary[source]['comments_without_saved_thread_post'] = len(missing_threads)
        summary[source]['comments_without_saved_direct_parent'] = len(missing_parents)
    print(source,json.dumps(summary[source],indent=2))


natmedpro {
  "records": 10,
  "passages": 1179,
  "record_types": {
    "monograph": 10
  },
  "duplicate_text_groups": 0,
  "records_in_duplicate_text_groups": 0,
  "source_titles": [
    "Ashwagandha",
    "Vitamin B12",
    "Turmeric",
    "Creatine",
    "Melatonin",
    "Magnesium",
    "Lactobacillus acidophilus",
    "Vitamin D",
    "Fish Oil",
    "Iron"
  ],
  "inline_reference_gaps": 55
}
reddit {
  "records": 4880,
  "passages": 5071,
  "record_types": {
    "comment": 4593,
    "post": 287
  },
  "duplicate_text_groups": 24,
  "records_in_duplicate_text_groups": 230,
  "mention_scope": {
    "thread_context": 3595,
    "own_text": 1285
  },
  "subreddits": {
    "Supplements": 2426,
    "herbalism": 2454
  },
  "comments_without_saved_thread_post": 60,
  "comments_without_saved_direct_parent": 53
}


## 5. Audit upstream clinical annotations, HTML and graph exports
Verify that clinical excerpt offsets point to current NatMed source text and HTML hashes agree. Count pending annotations without approving them. Check graph endpoint integrity; `MENTIONS` edges do not constitute clinical relationships.


In [6]:
natmed_by_id = {r['id']:r for r in source_rows['natmedpro']}
claims = read_jsonl(SOURCE_DIRS['natmedpro']/'upstream_derived/pending_clinical/clinical_claims.jsonl')
check_records('natmed_claims',claims,['id','claim_type','source','review'])
bad_claims = []
for claim in claims:
    try:
        ref=claim['source'];record=natmed_by_id[ref['record_id']]
        section=record['sections'][ref['section_index']]
        valid=(ref['content_hash']==record['content_hash']
               and section['text'][ref['start']:ref['end']]==ref['excerpt'])
    except (KeyError,IndexError,TypeError):valid=False
    if not valid:bad_claims.append(claim.get('id'))
finding('natmed_claims','error','invalid_claim_source_excerpt',len(bad_claims),bad_claims)
review_states=Counter(c.get('review',{}).get('state','<missing>') for c in claims)
finding('natmed_claims','review','pending_clinical_annotations',review_states.get('pending',0))
html_rows=read_jsonl(SOURCE_DIRS['natmedpro']/'source_html.jsonl')
bad_html=[r['record_id'] for r in html_rows if r['record_id'] not in natmed_by_id
          or hashlib.sha256(r['html'].encode()).hexdigest()!=r['sha256']
          or natmed_by_id[r['record_id']]['source_html_sha256']!=r['sha256']]
finding('natmedpro','error','invalid_html_binding',len(bad_html),bad_html)
summary['natmed_claims']={'count':len(claims),'review_states':dict(review_states),
                        'claim_types':dict(Counter(c['claim_type'] for c in claims))}
for source in ('natmedpro','reddit'):
    path=SOURCE_DIRS[source]/'upstream_derived'
    nodes=read_jsonl(path/'kg_nodes.jsonl');edges=read_jsonl(path/'kg_edges.jsonl')
    check_records(source+'_graph',nodes,['id','label'])
    ids={r['id'] for r in nodes}
    broken=[i for i,e in enumerate(edges) if e['source'] not in ids or e['target'] not in ids]
    finding(source,'error','graph_edges_with_missing_endpoints',len(broken),broken)
    summary[source]['upstream_graph']={'nodes':len(nodes),'edges':len(edges),
                                      'edge_types':dict(Counter(e['type'] for e in edges))}
print(json.dumps(summary['natmed_claims'],indent=2))


{
  "count": 1269,
  "review_states": {
    "pending": 1269
  },
  "claim_types": {
    "safety": 107,
    "adverse_effect": 120,
    "effectiveness": 809,
    "dose": 85,
    "interaction": 148
  }
}


## 6. Audit findings and raw-file immutability
`error` findings indicate failed integrity/structure checks; `review` findings identify decisions or qualifications needed before downstream use. Count totals across different checks must not be interpreted as numbers of distinct bad records. A clean integrity result does not mean all review issues are resolved.


In [7]:
raw_after=raw_snapshot()
raw_changes=sorted(k for k in raw_before.keys() | raw_after.keys() if raw_before.get(k)!=raw_after.get(k))
finding('all_sources','error','raw_files_changed_during_audit',len(raw_changes),raw_changes)
audit_report={'started_at_utc':audit_started_at,'finished_at_utc':datetime.now(timezone.utc).isoformat(),
              'sources':source_inventory,'summary':summary,'findings':findings,
              'raw_files_unchanged':not raw_changes,
              'mechanical_checks_passed':not any(f['severity']=='error' for f in findings),
              'preprocessing_performed':False}
print('Mechanical checks passed:',audit_report['mechanical_checks_passed'])
print('Raw files unchanged:',audit_report['raw_files_unchanged'])
print('Findings:')
for f in findings:
    print(f"{f['severity'].upper():6} {f['source']:16} {f['check']}: {f['count']}")
print('Audit report available in memory as audit_report. No dataset files were written.')


Mechanical checks passed: True
Raw files unchanged: True
Findings:
REVIEW dsld             off_market_labels: 231
REVIEW dsld             repeated_brand_product_name_groups: 70
REVIEW pubchem          no_match_queries: 511
REVIEW pubchem          missing_iupac_name: 3
REVIEW pubchem          ambiguous_candidate_queries: 7
REVIEW rxnorm           no_match_queries: 426
REVIEW rxnorm           normalized_match_queries: 31
REVIEW rxnorm           ambiguous_candidate_queries: 31
REVIEW mesh             no_match_queries: 469
REVIEW mesh             ambiguous_candidate_queries: 6
REVIEW natmedpro        monographs_requiring_source_or_clinical_review: 10
REVIEW natmedpro        inline_reference_numbers_missing_from_bibliographies: 55
REVIEW reddit           exact_duplicate_text_groups: 24
REVIEW reddit           comments_without_saved_thread_post: 60
REVIEW reddit           comments_without_saved_direct_parent: 53
REVIEW natmed_claims    pending_clinical_annotations: 1269
Audit report availabl

## 7. Preprocessing plan — decisions pending

Review the findings before implementing transformations. The next version of this notebook can produce derived tables under `data/processed/` after the target fields and rules are agreed.

| Source | Decisions to make after auditing |
|---|---|
| DSLD | Preserve label versions and market status; represent nested ingredients, forms, serving context and quantities without inventing conversions. |
| PubChem / RxNorm / MeSH | Review ambiguous and normalized candidates; maintain unmatched terms; distinguish ingredient categories, chemical forms, botanicals and organisms. |
| NatMed Pro | Keep source qualifications, bibliography gaps and review status; decide which pending annotations enter a research graph and how their provenance is represented. |
| Reddit | Review duplicate/boilerplate content, inherited topics and missing conversation context; assess de-identification needs before using text in evaluation. |
| All sources | Agree on stable project IDs, source-to-schema mappings, output tables and validation rules; keep evidence roles distinct. |

**Only the DSLD label-inventory step below is implemented. All other preprocessing remains pending.** Do not replace raw files, mark pending claims approved, treat no-match results as absent risks, or merge candidates solely on their returned names. Once rules are agreed, pilot the derived outputs on Magnesium before expanding to all ten supplement categories.


## 8. Preprocessing step 1 — DSLD label status and repeated names

Rules for this step:
- Keep all source labels and their original content. Do not merge labels sharing a brand/product name.
- Use source label IDs (`dsld:label:<id>`) as stable record IDs, not as canonical product identities.
- Map integer `offMarket=0` to `on_market_in_source`, `1` to `off_market_in_source`, and other values to `unknown`. This describes the saved snapshot, not verified current availability.
- Group identical, nonempty brand/product-name pairs for review. No fuzzy matching or deduplication.
- Preserve nested ingredients, forms, serving sizes and quantities unchanged inside `raw_label`.

Outputs: `label_inventory.jsonl` (all labels), `on_market_label_index.csv` (source-status index), `repeated_name_groups.json`, `validation_report.json`, and `manifest.json`. Input hashes and a rules version identify the output directory. Repeated runs reuse identical outputs and reject modified ones.

This cell intentionally writes **derived files only**, after the audit's mechanical checks pass. It does not apply the remaining proposed rules or change raw data.


In [8]:
if not audit_report['mechanical_checks_passed']:
    raise RuntimeError('Resolve audit errors before running preprocessing.')

import importlib.util
script_path = PROJECT_ROOT / 'scripts/preprocess_dsld.py'
spec = importlib.util.spec_from_file_location('preprocess_dsld', script_path)
dsld_preprocessor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dsld_preprocessor)
processed_dsld_dir, dsld_preprocessing_report = dsld_preprocessor.preprocess(PROJECT_ROOT)

raw_after_processing = raw_snapshot()
if raw_after_processing != raw_before:
    raise RuntimeError('Raw file fingerprints changed during this run; investigate before continuing.')
print('Processed output:', processed_dsld_dir.relative_to(PROJECT_ROOT))
print(json.dumps(dsld_preprocessing_report, indent=2))
print('All raw file fingerprints unchanged. Other sources have not been preprocessed.')


Processed output: data/processed/dsld/dsld-label-inventory-v1-b1689b0c095fb6ed
{
  "rules_version": "dsld-label-inventory-v1",
  "source_label_count": 400,
  "output_label_count": 400,
  "market_status_counts": {
    "off_market_in_source": 231,
    "on_market_in_source": 169
  },
  "repeated_name_groups": 70,
  "labels_in_repeated_name_groups": 206,
  "records_removed": 0,
  "records_merged": 0,
  "raw_label_content_preserved": true,
  "market_status_policy": "Preserve all labels. On-market index means offMarket=0 in this source snapshot, not verified current availability.",
  "repeated_name_policy": "Exact nonempty brand/product names flag review groups, never identity or automatic deduplication.",
  "ingredient_and_dose_processing": "Not performed; original fields remain inside raw_label."
}
All raw file fingerprints unchanged. Other sources have not been preprocessed.


## 9. Review repeated DSLD names without merging labels

Compare all 70 groups (206 labels; 304 pairs), preserving exact field values and DSLD relationship annotations. Differences in ingredientRows may reflect names, order, forms, quantities, or daily-value metadata—not necessarily reformulation. DSLD entry dates do not prove revision order. Identity decisions remain pending.

The generated `review.md` provides a table of all groups; `review.json` provides per-label details, source hashes, and pair comparisons. No records are removed or merged.


In [ ]:
import subprocess
import sys
review_run = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts/review_dsld_name_groups.py')],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
)
print(review_run.stdout)
if raw_snapshot() != raw_before:
    raise RuntimeError('Raw file fingerprints changed during review.')


/Users/jovian/Documents/ASU/ASU_COLLEGE_RELATED/ASU STUDY FOLDER/3rd Semester/SemanticWebMining/Project/SupplementRx AI-Conversational Agent/data/processed/dsld/dsld-name-review-v1-0baa34b90a1fe69e
{
  "group_count": 70,
  "label_count": 206,
  "pair_count": 304,
  "groups_with_field_differences": {
    "entryDate": 61,
    "events": 65,
    "ingredientRows": 54,
    "otheringredients": 49,
    "productVersionCode": 65,
    "statements": 68,
    "netContents": 40,
    "offMarket": 26,
    "upcSku": 32,
    "servingSizes": 23,
    "physicalState": 3
  },
  "groups_with_source_relationship_type": {
    "Image difference, same product": 51,
    "Package size difference, same product": 31,
    "Flavor difference": 2
  },
  "groups_without_internal_source_relationship": 8,
  "groups_with_multiple_on_market_labels": 16,
  "pairs_equal_except_id_and_src": 0,
  "pairs_same_nonempty_upc_sku": 126,
  "records_merged": 0,
  "records_removed": 0,
  "limitations": [
    "Field comparisons use exact

## 10. Ingredient and serving differences within repeated-name groups

Run `scripts/review_dsld_ingredients.py` to compare all 304 label pairs. The comparison preserves nested ingredient membership, forms, quantities, operators and serving basis; it ignores list ordering and separates daily-value metadata. Only whitespace is normalized. No ingredient synonyms or units are converted, and no product identities are assigned.

**Results:** 27 groups match on the compared ingredient/serving fields across all pairs; 43 groups have at least one difference. Matching these fields does not establish identical formulations: ingredient notes, other ingredients and source metadata remain separate. The group counts overlap: 21 name/hierarchy differences, 20 form differences, 20 amount/basis differences, 16 serving-definition differences, and 20 daily-value differences.

**Examples examined against the raw records:**

- BPI Sports Creatine Alkaline (25731 / 43261): 1500 mg versus 1.5 Gram(s), both per 2 capsules. The mass amounts are equivalent by arithmetic; the stored unit representation differs. No conversion is written to the data.
- Jarrow Formulas Magnesium Optimizer (7841 / 212304): 100 mg magnesium per 1 versus 2 tablets. The suggested-use text also lists 1 versus 2 tablets. Preserve the serving basis with the amount; do not merge these dose statements.
- InnovixLabs Triple Strength Omega-3 (248198 / 276213): DHA 288 versus 360 mg and EPA 552 versus 480 mg, each per 1 capsule. This is a substantive difference in the recorded amounts; it does not independently prove a manufacturing change.
- BetterYou Iron Daily Oral Spray (228638 / 257287): both list 5 mg iron, but the recorded serving basis is 0.52 versus 52 mL. Label 257287 is internally inconsistent: its serving note says `4 sprays (approx 0.02 fl. oz [0.52 ml])`, while the numeric serving fields say 52. Flag for source-label verification; do not silently correct it or calculate a normalized dose from it.

The detailed report is in `data/processed/dsld/dsld-ingredient-review-v1-0baa34b90a1fe69e/`. Ingredient-name changes are not automatically classified as added/removed biological ingredients. Source order does not establish chronology. All labels remain separate and raw files unchanged.


In [ ]:
ingredient_review_run = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts/review_dsld_ingredients.py')],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
)
print(ingredient_review_run.stdout)
if raw_snapshot() != raw_before:
    raise RuntimeError('Raw file fingerprints changed during ingredient review.')


/Users/jovian/Documents/ASU/ASU_COLLEGE_RELATED/ASU STUDY FOLDER/3rd Semester/SemanticWebMining/Project/SupplementRx AI-Conversational Agent/data/processed/dsld/dsld-ingredient-review-v1-0baa34b90a1fe69e
{
  "groups": 70,
  "labels": 206,
  "pairs": 304,
  "groups_all_pairs_same_compared_core": 27,
  "groups_with_core_difference": 43,
  "pair_status_counts": {
    "core_difference_needs_review": 157,
    "same_compared_core_identity_unresolved": 147
  },
  "group_flag_counts": {
    "daily_value_difference": 20,
    "ingredient_name_or_hierarchy_difference": 21,
    "ingredient_notes_or_description_difference": 50,
    "other_ingredients_raw_difference": 49,
    "form_difference": 20,
    "amount_or_basis_difference": 20,
    "serving_definition_difference": 16,
    "serving_notes_difference": 16,
    "ambiguous_repeated_ingredient_path": 1
  },
  "pair_flag_counts": {
    "daily_value_difference": 105,
    "ingredient_name_or_hierarchy_difference": 104,
    "ingredient_notes_or_descri

## 11. Inspect the actual DSLD example records

The following cells load the eight original JSON files directly and show each pair side by side. Expand **Full original JSON** below each table to see the complete record. No fields are corrected, converted or saved back to raw data.

Left/right order is for comparison, not a verified revision sequence. `offMarket` and `entryDate` describe the source snapshot. Full dictionaries are also available as `example_records["257287"]` (or any displayed label ID).


In [ ]:
import html
import json
from pathlib import Path
from IPython.display import HTML, display

# This section also runs independently from the project root or notebooks folder.
example_root = Path.cwd()
if not (example_root / 'data/raw/dsld/labels').is_dir():
    example_root = example_root.parent
if not (example_root / 'data/raw/dsld/labels').is_dir():
    raise FileNotFoundError('Run from the project root or notebooks directory.')

example_records = {}  # Full, unmodified records remain accessible by label ID.

def show_dsld_example(left_id, right_id):
    records = []
    for label_id in (left_id, right_id):
        source = example_root / 'data/raw/dsld/labels' / f'label_{label_id}.json'
        record = json.loads(source.read_text())
        assert str(record['id']) == str(label_id)
        example_records[str(label_id)] = record
        records.append((source.relative_to(example_root), record))

    def cell(value):
        return '<pre style="white-space:pre-wrap;overflow-wrap:anywhere;margin:0">' + html.escape(json.dumps(value, ensure_ascii=False, indent=2)) + '</pre>'

    def ingredient_rows(rows, parent=()):
        result = []
        for row in rows:
            path = parent + (row.get('name'),)
            result.append({'ingredient_path': list(path), 'forms': row.get('forms'),
                           'quantity': row.get('quantity'), 'notes': row.get('notes')})
            result.extend(ingredient_rows(row.get('nestedRows', []), path))
        return result

    def evidence(record):
        return {
            'Source identity': {k: record.get(k) for k in
                                ['id', 'brandName', 'fullName', 'upcSku', 'entryDate', 'offMarket']},
            'Serving definitions (including notes)': record.get('servingSizes'),
            'Ingredients: original quantities, units, serving basis, forms and daily values': ingredient_rows(record.get('ingredientRows', [])),
            'Suggested use / directions': [s for s in record.get('statements', [])
                                          if 'direction' in s.get('type', '').lower()],
            'Package contents': record.get('netContents'),
        }

    left, right = [evidence(record) for _, record in records]
    parts = ['<table style="width:100%;table-layout:fixed;border-collapse:collapse">',
             '<thead><tr><th style="width:20%">Source field</th>' +
             ''.join('<th>Label ' + str(record['id']) + '</th>' for _, record in records) + '</tr></thead><tbody>']
    parts.append('<tr><td>Raw file</td>' + ''.join('<td>' + html.escape(str(source)) + '</td>' for source, _ in records) + '</tr>')
    for title in left:
        parts.append('<tr><th style="vertical-align:top;text-align:left">' + html.escape(title) + '</th>' +
                     '<td style="vertical-align:top">' + cell(left[title]) + '</td>' +
                     '<td style="vertical-align:top">' + cell(right[title]) + '</td></tr>')
    parts.append('</tbody></table>')
    for source, record in records:
        parts.append('<details><summary>Full original JSON — label ' + str(record['id']) + '</summary>' + cell(record) + '</details>')
    display(HTML(''.join(parts)))


### BPI Sports — Creatine Alkaline

Compare Creatine quantities: **1500 mg** and **1.5 Gram(s)**, both per **2 capsules**. The mass amounts are equivalent by arithmetic; the raw representations are preserved.


In [ ]:
show_dsld_example(25731, 43261)


Source field,Label 25731,Label 43261
Raw file,data/raw/dsld/labels/label_25731.json,data/raw/dsld/labels/label_43261.json
Source identity,"{ ""id"": 25731, ""brandName"": ""BPI Sports"", ""fullName"": ""Creatine Alkaline"", ""upcSku"": ""8 51780 00591 0"", ""entryDate"": ""2013-09-25"", ""offMarket"": 1 }","{ ""id"": 43261, ""brandName"": ""BPI Sports"", ""fullName"": ""Creatine Alkaline"", ""upcSku"": ""8 51780 00591 0"", ""entryDate"": ""2015-03-25"", ""offMarket"": 0 }"
Serving definitions (including notes),"[ { ""order"": 1, ""minQuantity"": 2, ""maxQuantity"": 2, ""minDailyServings"": 1, ""maxDailyServings"": 1, ""unit"": ""Capsule(s)"", ""inSFB"": true } ]","[ { ""order"": 1, ""minQuantity"": 2, ""maxQuantity"": 2, ""minDailyServings"": 1, ""maxDailyServings"": 1, ""unit"": ""Capsule(s)"", ""inSFB"": true } ]"
"Ingredients: original quantities, units, serving basis, forms and daily values","[ { ""ingredient_path"": [ ""Creatine"" ], ""forms"": [ { ""order"": 1, ""ingredientId"": 6225, ""prefix"": ""and"", ""percent"": null, ""name"": ""Creatine HCl"", ""category"": ""non-nutrient/non-botanical"", ""ingredientGroup"": ""Creatine"", ""uniiCode"": null }, { ""order"": 2, ""ingredientId"": 4475, ""prefix"": ""as pH-Sorb(TM) Buffering Technology"", ""percent"": null, ""name"": ""Creatine Monohydrate"", ""category"": ""non-nutrient/non-botanical"", ""ingredientGroup"": ""Creatine"", ""uniiCode"": null } ], ""quantity"": [ { ""servingSizeOrder"": 1, ""servingSizeQuantity"": 2, ""operator"": ""="", ""quantity"": 1500, ""unit"": ""mg"", ""dailyValueTargetGroup"": [ { ""name"": ""Adults and children 4 or more years of age"", ""operator"": null, ""percent"": null, ""footnote"": ""Daily Value not established"" } ], ""servingSizeUnit"": ""Capsule(s)"" } ], ""notes"": ""Creatine (Form: as pH-Sorb(TM) Buffering Technology Creatine Monohydrate, and Creatine HCl) Note: patent-pending "" } ]","[ { ""ingredient_path"": [ ""Creatine"" ], ""forms"": [ { ""order"": 1, ""ingredientId"": 6225, ""prefix"": ""and"", ""percent"": null, ""name"": ""Creatine HCl"", ""category"": ""non-nutrient/non-botanical"", ""ingredientGroup"": ""Creatine"", ""uniiCode"": null }, { ""order"": 2, ""ingredientId"": 4475, ""prefix"": ""as pH-Sorb(TM) Buffering Technology"", ""percent"": null, ""name"": ""Creatine Monohydrate"", ""category"": ""non-nutrient/non-botanical"", ""ingredientGroup"": ""Creatine"", ""uniiCode"": null } ], ""quantity"": [ { ""servingSizeOrder"": 1, ""servingSizeQuantity"": 2, ""operator"": ""="", ""quantity"": 1.5, ""unit"": ""Gram(s)"", ""dailyValueTargetGroup"": [ { ""name"": ""Adults and children 4 or more years of age"", ""operator"": null, ""percent"": null, ""footnote"": ""Daily Value not established"" } ], ""servingSizeUnit"": ""Capsule(s)"" } ], ""notes"": ""Creatine (Form: as pH-Sorb(TM) Buffering Technology Creatine Monohydrate, and Creatine HCl) Note: patent-pending "" } ]"
Suggested use / directions,"[ { ""type"": ""Suggested/Recommended/Usage/Directions"", ""notes"": ""Suggested Use: Take 1 serving (2 capsules) with 12 ounces of water once daily on an empty stomach or with a meal, or as directed by a qualified healthcare practitioner."" } ]","[ { ""type"": ""Suggested/Recommended/Usage/Directions"", ""notes"": ""Suggested Use: Take one (1) serving daily, before and or after training, or as directed by a qualified healthcare practitioner. Can be taken with or without food."" } ]"
Package contents,"[ { ""order"": 1, ""quantity"": 120, ""unit"": ""Capsule(s)"", ""display"": ""120 Capsule(s)"" }, { ""order"": 2, ""quantity"": 60, ""unit"": ""Serving(s)"", ""display"": ""60 Serving(s)"" } ]","[ { ""order"": 1, ""quantity"": 120, ""unit"": ""Capsule(s)"", ""display"": ""120 Capsule(s)"" } ]"


### Jarrow Formulas — Magnesium Optimizer

Compare Magnesium: **100 mg per 1 tablet** versus **100 mg per 2 tablets**. Check both `servingSizeQuantity` in the ingredient quantities and the serving definitions. Suggested-use text also differs.


In [ ]:
show_dsld_example(7841, 212304)


DSLD raw-label comparison (HTML table and expandable original JSON)

### InnovixLabs — Triple Strength Omega-3

Look inside the nested Omega-3 ingredient rows: **DHA 288 versus 360 mg** and **EPA 552 versus 480 mg**, each per **1 capsule**. These are different recorded amounts, not proof of a manufacturing change.


In [ ]:
show_dsld_example(248198, 276213)


DSLD raw-label comparison (HTML table and expandable original JSON)

### BetterYou — Iron Daily Oral Spray

Both records list **5 mg iron**. Compare the serving basis: **0.52 mL** versus **52 mL**. In label **257287**, the numeric serving fields conflict with its own serving note, which says **0.52 ml**. This remains unresolved; no value has been corrected.


In [ ]:
show_dsld_example(228638, 257287)


Source field,Label 228638,Label 257287
Raw file,data/raw/dsld/labels/label_228638.json,data/raw/dsld/labels/label_257287.json
Source identity,"{ ""id"": 228638, ""brandName"": ""BetterYou"", ""fullName"": ""Iron Daily Oral Spray"", ""upcSku"": """", ""entryDate"": ""2020-06-24"", ""offMarket"": 1 }","{ ""id"": 257287, ""brandName"": ""BetterYou"", ""fullName"": ""Iron Daily Oral Spray"", ""upcSku"": """", ""entryDate"": ""2021-12-18"", ""offMarket"": 1 }"
Serving definitions (including notes),"[ { ""order"": 1, ""minQuantity"": 0.52, ""maxQuantity"": 0.52, ""minDailyServings"": 1, ""maxDailyServings"": 1, ""unit"": ""mL"", ""notes"": ""(4 sprays) (0.02 fl. oz)"", ""inSFB"": true } ]","[ { ""order"": 1, ""minQuantity"": 52, ""maxQuantity"": 52, ""minDailyServings"": 1, ""maxDailyServings"": 1, ""unit"": ""mL"", ""notes"": ""4 sprays (approx 0.02 fl. oz [0.52 ml])"", ""inSFB"": true } ]"
"Ingredients: original quantities, units, serving basis, forms and daily values","[ { ""ingredient_path"": [ ""Iron"" ], ""forms"": [ { ""order"": 1, ""ingredientId"": 227262, ""prefix"": ""and"", ""percent"": null, ""name"": ""Ferric Ammonium Citrate"", ""category"": ""other"", ""ingredientGroup"": ""Ferric Ammonium Citrate"", ""uniiCode"": null }, { ""order"": 2, ""ingredientId"": 252967, ""prefix"": ""as"", ""percent"": null, ""name"": ""Ferric Sodium Edetate"", ""category"": ""TBD"", ""ingredientGroup"": ""TBD"", ""uniiCode"": null } ], ""quantity"": [ { ""servingSizeOrder"": 1, ""servingSizeQuantity"": 0.52, ""operator"": ""="", ""quantity"": 5, ""unit"": ""mg"", ""dailyValueTargetGroup"": [ { ""name"": ""Adults and children 4 or more years of age"", ""operator"": ""="", ""percent"": 28, ""footnote"": null } ], ""servingSizeUnit"": ""mL"" } ], ""notes"": ""Iron (Form: as Ferric Sodium Edetate, and Ferric Ammonium Citrate) "" } ]","[ { ""ingredient_path"": [ ""Iron"" ], ""forms"": [ { ""order"": 1, ""ingredientId"": 227262, ""prefix"": ""and"", ""percent"": null, ""name"": ""Ferric Ammonium Citrate"", ""category"": ""other"", ""ingredientGroup"": ""Ferric Ammonium Citrate"", ""uniiCode"": null }, { ""order"": 2, ""ingredientId"": 252967, ""prefix"": ""as"", ""percent"": null, ""name"": ""Ferric Sodium Edetate"", ""category"": ""TBD"", ""ingredientGroup"": ""TBD"", ""uniiCode"": null } ], ""quantity"": [ { ""servingSizeOrder"": 1, ""servingSizeQuantity"": 52, ""operator"": ""="", ""quantity"": 5, ""unit"": ""mg"", ""dailyValueTargetGroup"": [ { ""name"": ""Adults and children 4 or more years of age"", ""operator"": ""="", ""percent"": 28, ""footnote"": null } ], ""servingSizeUnit"": ""mL"" } ], ""notes"": ""Iron (Form: as Ferric Sodium Edetate, and Ferric Ammonium Citrate) "" } ]"
Suggested use / directions,"[ { ""type"": ""Suggested/Recommended/Usage/Directions"", ""notes"": ""Directions: Shake well before every use. Carefully spray into your mouth, taste and swallow."" } ]","[ { ""type"": ""Suggested/Recommended/Usage/Directions"", ""notes"": ""Directions: Shake well before every use. Carefully spray into your mouth, taste and swallow."" } ]"
Package contents,"[ { ""order"": 1, ""quantity"": 0.85, ""unit"": ""Fluid Ounce(s)"", ""display"": ""0.85 Fluid Ounce(s)"" }, { ""order"": 2, ""quantity"": 25, ""unit"": ""mL"", ""display"": ""25 mL"" } ]","[ { ""order"": 1, ""quantity"": 0.85, ""unit"": ""Fluid Ounce(s)"", ""display"": ""0.85 Fluid Ounce(s)"" }, { ""order"": 2, ""quantity"": 25, ""unit"": ""mL"", ""display"": ""25 mL"" } ]"


## 12. Dataset-wide quantity and serving audit

This step audits **all 400 labels**, including nested ingredients, without correcting or converting data. The script checks amount types, unit presence, operators, serving references, serving-size ranges, missing daily-serving bounds, and explicit mL values in mL serving notes.

| Finding | Entries / occurrences | Distinct labels |
|---|---:|---:|
| `NP` unit, always stored with quantity zero | 705 | 74 |
| Missing ingredient quantity serving basis | 8 | 2 |
| Incomplete daily-serving ranges | 35 | 35 |
| Quantity basis outside stated per-serving range | 67 | 9 |
| Numeric mL serving conflicts with mL note | 1 | 1 |
| Zero quantity with a named unit (informational) | 32 | 14 |
| Inequality operator (informational) | 24 | 14 |

Counts overlap. There are **2,339 ingredient rows**, **2,392 quantity entries**, and **400 serving definitions**. No rows lack quantity entries; no amount values are nonnumeric, nonfinite or negative; serving references resolve, and ingredient serving-unit text matches the referenced serving-unit text. These checks do not establish that all doses are accurate.

**Interpretation:**

- `NP` semantics remain unverified. Do not interpret these 705 zeros as confirmed zero ingredient content or impute a dose.
- Of 35 incomplete daily-serving ranges, 33 lack the maximum and 2 lack both bounds. Do not infer daily frequency from a missing bound.
- Eight entries on labels 73905 and 214296 lack `servingSizeQuantity`. Preserve the missing value; normalization needs a separately justified rule.
- Of 67 out-of-range quantity bases, **52 fall within the range computed from serving size × daily frequency**. This suggests an alternate daily-total basis, not automatic permission to relabel them. Fifteen do not satisfy that test (including when daily bounds are unavailable).
- BetterYou label 257287 has 52 mL in numeric fields and 0.52 ml in its serving note. Leave unresolved pending original-label verification.
- Preserve inequality operators. Named-unit zeros may be legitimate declarations and are not automatically missing values.
- The scan does not exhaustively interpret free text, verify images, validate daily values or add nested blend components. Different unit families are not automatically converted.

The next cells rerun the audit and display one actual raw-data example for every finding category. `quantity_audit_report['findings']` contains all findings with label IDs and JSON pointers.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

quantity_audit_root = Path.cwd()
if not (quantity_audit_root / 'data/raw/dsld').is_dir():
    quantity_audit_root = quantity_audit_root.parent
quantity_audit_run = subprocess.run(
    [sys.executable, str(quantity_audit_root / 'scripts/audit_dsld_quantities.py')],
    cwd=quantity_audit_root, check=True, capture_output=True, text=True,
)
quantity_audit_dir = Path(quantity_audit_run.stdout.splitlines()[0])
quantity_audit_report = json.loads((quantity_audit_dir / 'audit.json').read_text())
print('Report:', quantity_audit_dir.relative_to(quantity_audit_root))
print(json.dumps(quantity_audit_report['summary'], indent=2))


Report: data/processed/dsld/dsld-quantity-audit-v1-ae1a24aa4a66389a
{
  "labels": 400,
  "ingredient_rows": 2339,
  "quantity_entries": 2392,
  "nested_ingredient_rows": 786,
  "np_entries": 705,
  "np_zero_entries": 705,
  "serving_definitions": 400,
  "quantity_unit_counts": {
    "mg": 1060,
    "NP": 705,
    "mcg": 204,
    "IU": 56,
    "{Calories}": 26,
    "g": 8,
    "ug": 1,
    "Gram(s)": 213,
    "DU": 3,
    "HUT": 3,
    "FCCFIP": 3,
    "CU": 3,
    "INVU": 3,
    "ALU": 3,
    "DP": 6,
    "GaIU": 3,
    "Endo-PGU": 3,
    "XU": 3,
    "HCU": 3,
    "BGU": 3,
    "Kcal": 1,
    "Calorie(s)": 62,
    "grams": 2,
    "mL": 1,
    "mcg DFE": 12,
    "mcg RAE": 1,
    "mg NE": 1
  },
  "serving_unit_counts": {
    "Capsule(s)": 162,
    "Tablet(s)": 82,
    "mL": 24,
    "Gram(s)": 25,
    "Softgel(s)": 29,
    "Gummy(ies)": 2,
    "tsp": 7,
    "Caplet(s)": 10,
    "Vegicap(s)": 1,
    "Soft Chew(s)": 4,
    "White Tablet(s)": 1,
    "Drop(s)": 3,
    "Vegetarian Capsule(s

In [ ]:
quantity_audit_examples = {}
for finding in quantity_audit_report['findings']:
    quantity_audit_examples.setdefault(finding['code'], finding)
for code, finding in sorted(quantity_audit_examples.items()):
    source_path = quantity_audit_root / 'data/raw/dsld/labels' / f"label_{finding['label_id']}.json"
    original_label = json.loads(source_path.read_text())
    original_value = original_label
    for token in finding['pointer'].strip('/').split('/'):
        original_value = original_value[int(token)] if isinstance(original_value, list) else original_value[token]
    print(f"\n{code}: {finding['brand']} / {finding['product']} (label {finding['label_id']})")
    print('Source:', source_path.relative_to(quantity_audit_root), finding['pointer'])
    print(json.dumps({'raw_value_at_pointer': original_value, 'audit_context': finding['context'],
                      'label_serving_definitions': original_label['servingSizes']}, ensure_ascii=False, indent=2))



incomplete_daily_serving_range: Vitamer Laboratories / Turmeric Factors (label 23009)
Source: data/raw/dsld/labels/label_23009.json /servingSizes/0
{
  "raw_value_at_pointer": {
    "order": 1,
    "minQuantity": 1,
    "maxQuantity": 1,
    "minDailyServings": 1,
    "maxDailyServings": null,
    "unit": "Tablet(s)",
    "inSFB": true
  },
  "audit_context": null,
  "label_serving_definitions": [
    {
      "order": 1,
      "minQuantity": 1,
      "maxQuantity": 1,
      "minDailyServings": 1,
      "maxDailyServings": null,
      "unit": "Tablet(s)",
      "inSFB": true
    }
  ]
}

inequality_amount: SomaLife International / Omega-3 (label 33371)
Source: data/raw/dsld/labels/label_33371.json /ingredientRows/0/quantity/0
{
  "raw_value_at_pointer": {
    "servingSizeOrder": 1,
    "servingSizeQuantity": 1,
    "operator": ">=",
    "quantity": 360,
    "unit": "mg",
    "dailyValueTargetGroup": [
      {
        "name": "Adults and children 4 or more years of age",
        "operat

## 13. DSLD baseline completion: structures, completeness, identifiers and source coverage

This complements sections 1–12. It profiles **213 fields across 31 object paths**, checks identifiers and additional duplicate candidates, and summarizes relationships, dates and text availability. It does not merge or correct records.

| Area | Finding | Schema implication |
|---|---|---|
| Completeness | 52 empty UPC/SKU values; 82 null servings-per-container; 68 product-version codes say `Not Present` | Preserve source states; do not treat every placeholder as interchangeable with null |
| Other ingredients | Lists on 387 labels; 13 null lists; free text on 1 label | Empty or absent data does not prove no other ingredients |
| Duplicate candidates | 71 normalized-name groups; 4 cross exact-name boundaries; 62 shared whitespace-compacted UPC/SKU groups, 10 crossing exact-name boundaries | Candidate evidence only; no new unique-product count or merging decision |
| Ingredient identifiers | 5,438 ingredient/form occurrences, including other ingredients; 1,847 distinct DSLD ingredient IDs; 291 normalized names map to multiple IDs | Names are not sufficient identifiers; source IDs need namespace and mapping review |
| UNII field | 382 occurrences contain `0` (224) or `1` (158) | Preserve raw values; do not accept as verified UNII identifiers |
| Label relationships | 1,739 entries but 1,720 distinct directed typed links; 916 entries point to 451 IDs outside the sample | Duplicate edges can be consolidated with provenance; outside-sample targets are not automatically broken links |
| Relationship consistency | No self-links; 5 internal directed links lack a reverse link of the same type | Record source direction; do not invent reciprocal edges |
| Dates | Entry dates span 2011-10-26 to 2025-09-25; no date parse or entry-event disagreement findings | Entry date is not manufacture, revision, expiry, or a currency guarantee |
| Label content | Directions on 398 labels; precautions on 366; allergy precaution text on 61; claim classification codes on 400 | Content availability does not establish effectiveness, complete safety coverage or absence of risk |

**Baseline status:** sufficient for drafting the shared schema, together with the preceding audits. Still pending: authoritative interpretation of source codes such as NP, external identifier verification, image/source accuracy checks, semantic identity resolution and clinical evidence review. These are not prerequisites for moving to the next dataset's baseline audit.

The following cells provide the summary, field-completeness details and inspectable examples. The full report and field profiles remain available as `dsld_baseline_report` and `dsld_field_profiles`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

baseline_root = Path.cwd()
if not (baseline_root / 'data/raw/dsld').is_dir():
    baseline_root = baseline_root.parent
baseline_run = subprocess.run(
    [sys.executable, str(baseline_root / 'scripts/audit_dsld_baseline.py')],
    cwd=baseline_root, check=True, capture_output=True, text=True,
)
baseline_dir = Path(baseline_run.stdout.splitlines()[0])
dsld_baseline_report = json.loads((baseline_dir / 'audit.json').read_text())
dsld_field_profiles = json.loads((baseline_dir / 'field_profiles.json').read_text())
print('Output:', baseline_dir.relative_to(baseline_root))
print(json.dumps(dsld_baseline_report['summary'], indent=2))


Output: data/processed/dsld/dsld-baseline-audit-v1-1487bf52f8e90a1f
{
  "labels": 400,
  "object_paths_profiled": 31,
  "field_profiles": 213,
  "normalized_name_candidate_groups": 71,
  "new_normalized_name_groups_beyond_exact_names": 4,
  "shared_compact_upc_sku_groups": 62,
  "shared_upc_sku_groups_crossing_exact_names": 10,
  "ingredient_occurrences": 5438,
  "distinct_ingredient_ids": 1847,
  "ingredient_ids_with_multiple_names": 0,
  "normalized_ingredient_names_with_multiple_ids": 291,
  "unii_shape_review_occurrences": 382,
  "upc_sku_present_labels": 348,
  "relationships": {
    "occurrences": 1739,
    "distinct_directed_relationships": 1720,
    "internal_occurrences": 823,
    "external_occurrences": 916,
    "distinct_external_targets": 451,
    "self_references": 0,
    "internal_edges_without_same_type_reverse": 5,
    "type_counts": {
      "Image difference, same product": 767,
      "Package size difference, same product": 775,
      "Bundle set": 53,
      "Flavor d

In [ ]:
print('TOP-LEVEL FIELD COMPLETENESS (all 400 label records)')
print(json.dumps(dsld_field_profiles['label'], indent=2))
print('OTHER-INGREDIENT FIELD COMPLETENESS')
print(json.dumps(dsld_field_profiles['label/otheringredients'], indent=2))
print('NAME CANDIDATES CROSSING EXACT-NAME GROUPS — NOT MERGE DECISIONS')
print(json.dumps([g for g in dsld_baseline_report['duplicate_candidates']['normalized_names']
                  if g['crosses_exact_name_groups']], ensure_ascii=False, indent=2))
print('SHARED UPC/SKU EXAMPLES CROSSING EXACT NAMES')
print(json.dumps([g for g in dsld_baseline_report['duplicate_candidates']['shared_compact_upc_sku']
                  if g['crosses_exact_name_groups']][:3], ensure_ascii=False, indent=2))
print('INGREDIENT NAMES ASSOCIATED WITH MULTIPLE SOURCE IDS — FIRST FIVE')
print(json.dumps(dsld_baseline_report['identifier_review']['normalized_names_with_multiple_ids'][:5], indent=2))
print('RAW UNII VALUES REQUIRING REVIEW — ONE EXAMPLE PER VALUE')
unii_examples = {}
for occurrence in dsld_baseline_report['identifier_review']['unii_nonempty_not_10_alphanumeric']:
    unii_examples.setdefault(str(occurrence['uniiCode']), occurrence)
for example in unii_examples.values():
    label = json.loads((baseline_root / 'data/raw/dsld/labels' / f"label_{example['label_id']}.json").read_text())
    value = label
    for token in example['pointer'].strip('/').split('/'):
        value = value[int(token)] if isinstance(value, list) else value[token]
    print(json.dumps({'label_id': example['label_id'], 'pointer': example['pointer'], 'raw_object':value}, ensure_ascii=False, indent=2))
print('EXAMPLE SOURCE DIRECTIONS AND ALLERGY PRECAUTIONS')
print(json.dumps({k:dsld_baseline_report['statement_examples'][k][:1]
                  for k in ['directions','allergy_precautions']}, ensure_ascii=False, indent=2))


TOP-LEVEL FIELD COMPLETENESS (all 400 label records)
{
  "brandIpSymbol": {
    "parent_objects": 400,
    "missing_key": 0,
    "states": {
      "populated": 399,
      "empty_string": 1
    },
    "types": {
      "str": 400
    },
    "placeholder_examples": []
  },
  "brandName": {
    "parent_objects": 400,
    "missing_key": 0,
    "states": {
      "populated": 400
    },
    "types": {
      "str": 400
    },
    "placeholder_examples": []
  },
  "bundleName": {
    "parent_objects": 400,
    "missing_key": 0,
    "states": {
      "empty_string": 384,
      "populated": 16
    },
    "types": {
      "str": 400
    },
    "placeholder_examples": []
  },
  "claims": {
    "parent_objects": 400,
    "missing_key": 0,
    "states": {
      "populated": 400
    },
    "types": {
      "list": 400
    },
    "placeholder_examples": []
  },
  "contacts": {
    "parent_objects": 400,
    "missing_key": 0,
    "states": {
      "populated": 400
    },
    "types": {
      "list": 400

## 14. DSLD POC schema draft — selected fields and review rules

**Status: design only, not implemented.** This is the DSLD portion of a future shared schema. Test and extend it against the other sources before finalizing the graph. Adding this cell does not normalize values, merge records, approve identifiers or create graph entities.

### Scope and intended questions

Support: “What does this product contain?”, “How much is listed per serving?”, “How do these labels compare?”, “What directions and precautions appear on the label?”, and “Which source supports this answer?” Label directions are manufacturer instructions, not personalized recommendations. Manufacturer statements do not independently establish clinical effectiveness or safety.

### Proposed entities, fields and relationships

| Entity | Selected source fields / derived fields | Identity and relationships |
|---|---|---|
| **LabelRecord** | `id`, `brandName`, `fullName`, `entryDate`, `offMarket`; source file/hash | Source identity `dsld:label:<id>`. A capture/version also records the source hash, so later captures do not overwrite earlier evidence. Label records remain separate. |
| **ProductCandidate** | Brand/name plus identity-review evidence | Provisional grouping, not an approved product identity. A label `CANDIDATE_FOR` a product candidate. Accepted product/formulation/package entities can be added after review; no name-only merging. |
| **Ingredient** | `ingredientId`, `name`; source IDs and original names | Namespace DSLD ingredient IDs. A name is not an identity key; different IDs with the same name remain mapping candidates. If an ID is absent, use a source-local occurrence identity until resolved. |
| **CompositionAssertion** | Parent/context, ingredient reference, role (`listed_ingredient` or `other_ingredient`), amount, unit, operator, serving basis, original `description`/`notes`, review state | A label `SUPPORTS` an assertion; assertion `ABOUT_INGREDIENT` an ingredient. An assertion retains its subject context and can `HAS_COMPONENT` another assertion to preserve nested membership. It is not a free-floating ingredient–amount pair. |
| **IngredientFormAssertion** | `forms`: form `ingredientId`, `name`, `prefix`, `percent`, original order | Attached to a particular composition assertion via `HAS_FORM`; references the form ingredient. Preserve prefixes/percentages, including missing values. Do not assign the parent amount to every form. |
| **ServingDefinition** | `servingSizes`: `order`, `minQuantity`, `maxQuantity`, `unit`, `minDailyServings`, `maxDailyServings`, `notes`, `inSFB` | Initially identified within a label capture by serving order. Composition quantities `USES_SERVING` the referenced definition but retain their own explicit serving quantity/unit. Preserve disagreements rather than overwriting either representation. |
| **LabelStatement** | Selected `statements`: `type`, `notes` | Preserve directions, every precaution category and storage instructions, linked to the source label. Identical text can share a text representation, but each source attribution and applicability remain explicit. |
| **SourceEvidence** | Label ID, file, hash, JSON pointer, original extracted values, extraction-rule version | Every extracted assertion links to evidence. Separate source occurrences may support an equivalent assertion; they do not count as independent clinical evidence. |

**Amount multiplicity:** Each ingredient row may contain several `quantity` entries. Retain one assertion per quantity/basis (or explicit child quantity assertions); never keep only the first. When quantity is absent or unusable, retain an ingredient-presence assertion with its unresolved amount status. Nested ingredient rows retain their own amounts and parent membership; do not double-count a blend total plus its components.

### Quantity mapping and types

| Source | Draft processed representation | Rule |
|---|---|---|
| `quantity[].quantity` | `original_amount`; optional `normalized_amount` | Preserve source numeric value. Use exact decimal handling for normalized arithmetic, not rounded comparison. |
| `quantity[].unit` | `original_unit`; optional `normalized_unit` | Use an explicit unit dictionary. Compatible mass aliases/conversions may be normalized; IU, activity units and nutrient equivalents require separate rules. |
| `quantity[].operator` | `operator` | Preserve equality and inequalities; `< 1` is not the same fact as `= 1`. |
| `quantity[].servingSizeOrder` | `source_serving_order` / serving reference | Resolve within the source label capture. |
| `quantity[].servingSizeQuantity`, `servingSizeUnit` | `basis_quantity`, `basis_unit` | Preserve explicit source basis, even if it differs from the serving definition. Missing basis remains unknown. |
| `description`, ingredient `notes`, serving `notes` | Source text plus evidence links | Retain qualifications and contradictions; normalized text must not silently erase them. |
| Source paths and hashes | `evidence_refs` | Keep every contributing source occurrence when equivalent assertions are consolidated. |
| Audit findings | `review_flags`, `normalization_status`, `normalization_rule_id` | Separate missing/unresolved/conflicting from valid zero or exact values. Flags are derived annotations, not source corrections. |

Identifiers are namespaced strings. Numeric amounts and ranges are nullable decimals; operators are controlled strings; source dates are ISO dates with original text retained in evidence. Preserve the distinction between missing key, explicit null, empty string/container and candidate placeholder in source-state metadata. Missing daily-serving bounds are not zero or “once daily.”

For `NP` with stored zero: retain both original values, leave normalized amount/unit unresolved, and flag `unresolved_unit`. This records uncertainty; it does not assert the ingredient has zero content. Preserve named-unit zeros and inequalities. For conflicts such as BetterYou, retain both source representations and prevent unsupported dose calculations; do not choose a corrected value.

### Supporting fields retained internally

- `upcSku`, `productVersionCode`, `labelRelationships`, `netContents`, `physicalState`: evidence for product/formulation/package review. UPC/SKU is not assumed unique; source relationships are assertions, not independently validated identity decisions.
- `otheringredients.text`, ingredient lists and their forms: preserve original text and structured composition separately. A missing list does not establish absence of other ingredients.
- Source ID/hash/pointer, original text/values, review flags and rule versions: mandatory provenance even when not shown in ordinary answers.
- Entry date and market status: display historical context when relevant; neither establishes current availability.

**Deferred from the first graph:** contacts, images, branding symbols, general marketing text, claim-classification codes and daily-value percentages. They remain in raw files. This scope can be expanded later without recollecting the current source records.

### Identity, normalization and deduplication rules

1. **Keep source labels and captures.** Normalization operates on derived values. No original label is deleted because its facts overlap with another label.
2. **Normalize only justified representations.** Whitespace, explicit unit aliases and exact compatible mass conversions can support equivalence. No ingredient synonym, salt/form equivalence, or product identity is inferred from names alone.
3. **Define assertion equivalence by context.** Compare established subject context, ingredient identity, ingredient role, nested/blend context, attached forms, operator, normalized amount/unit, explicit serving basis and relevant qualifications. Matching amounts alone are insufficient. Label-local serving IDs may differ; semantic serving context must match before cross-label consolidation.
4. **Keep equivalence separate from identity.** Until shared product/formulation context is established, label-scoped assertions can be linked as equivalent amount representations without being collapsed into one product fact. Later, equivalent assertions within an accepted context may share one assertion with multiple evidence links.
5. **Preserve differences that affect answers.** Different forms, amounts, bases, compositions or conflicting qualifications remain distinct assertions. Shared product identity does not mean identical formulation or package. Matching selected fields does not prove complete formulation equivalence.
6. **Represent conflict explicitly.** Mark affected assertions with `source_conflict`, preserve the conflicting evidence and leave normalization unresolved. Do not create an averaged or preferred dose. A conflict in amount does not invalidate ingredient-presence information.
7. **Deduplicate graph links without inflating evidence.** Exact duplicate directed typed links can be represented once with all source occurrences retained. Do not automatically add reverse links or treat repeated labels as independent support for effectiveness.

### Test the draft against the four inspected examples

| Example / label IDs | Expected representation | What must remain separate or unresolved |
|---|---|---|
| **BPI Creatine Alkaline — 25731 / 43261** | `1500 mg` and `1.5 Gram(s)` normalize to the same creatine amount per two capsules; preserve both original representations and evidence links. | Complete label identity is not established by the amount. Other ingredients differ, including colorant/opacifier entries on 43261. Share equivalent amount representation first; cross-label assertion merging requires established subject context. |
| **Jarrow Magnesium Optimizer — 7841 / 212304** | Preserve 100 mg magnesium with a one-tablet versus two-tablet basis, plus the respective directions. | These are distinct amount/basis assertions; do not merge because the number 100 matches. |
| **InnovixLabs Triple Strength Omega-3 — 248198 / 276213** | Preserve nested DHA/EPA assertions: DHA 288/360 mg and EPA 552/480 mg per capsule, attached to their source contexts. | Different recorded compositions remain distinct; do not assert a manufacturing change or revision chronology without evidence. |
| **BetterYou Iron Daily Oral Spray — 228638 / 257287** | Preserve 5 mg iron and all source serving information. Flag 257287's numeric 52 mL versus its note's 0.52 ml. | No automatic correction to 0.52 and no normalized dose calculation using the conflicting basis. |

**Acceptance criteria before implementation:** every selected assertion traces to a source pointer; quantities retain operator/basis and hierarchy; equivalent mass representations can be recognized without merging different product contexts; conflicting and missing data remain explicit; other-ingredient differences survive; all four examples can be represented without loss. Extend this draft with the selected fields and evidence roles from NatMed, Reddit, PubChem, RxNorm and MeSH before finalizing the shared schema.


## 15. PubChem baseline audit — structure, completeness, identities and source quality

**Scope:** offline audit of the saved 2026-10-28 PubChem snapshot. No new downloads, mapping approvals, corrections or merging. The same audit categories used for DSLD are applied here with chemistry/query-specific checks.

| Category | Finding | Interpretation |
|---|---|---|
| Structure/provenance | 739 query groups from 753 source-name variants; all 3,391 source occurrences verified (2,339 ingredient rows + 1,052 forms) across 400 labels | Every query occurrence resolves to the saved DSLD name/ID/path; other ingredients are outside the original collection scope |
| Search outcomes | 228 queries returned candidates: 221 single-candidate and 7 ambiguous; 511 returned no match | Single-candidate results remain unreviewed. No-match does not mean the ingredient does not exist in PubChem |
| Compound records | 239 CID-keyed records, 272 query-to-CID candidate links | Multiple queries can share a CID; a query can return multiple CIDs |
| Completeness | 3 missing IUPAC names; all seven other requested properties present (CID, title, formula, weight, connectivity SMILES, InChI, InChIKey) | Missing IUPACName is a field-level gap, not automatic rejection of the record |
| Identifier reuse | 30 CIDs are reached by multiple queries | Reuse compound entities by CID while retaining separate, unapproved ingredient mappings |
| Synonyms | 46,426 entries; no exact duplicates within a CID; 74 normalized synonyms shared across CIDs | Synonyms are not unique identity keys or automatic merge rules |
| Structure overlap | 8 groups share InChIKey; 8 share InChI; 13 share connectivity SMILES; 21 share molecular formula | Preserve CIDs and full structures; chemical equivalence needs review beyond string overlap |
| Text matching | 3 query/candidate pairs lack a whitespace/case-normalized title/synonym match | Examples differ in hyphenation; this flag does not establish a wrong result |
| Integrity | Collection hash/size, source-label hashes, query scope/paths, record CID consistency and assembled records versus batch responses pass | Traceability passes; chemical mapping validity remains unverified |

**Collection behavior:** six multi-CID responses have a different order in the derived CID list because the collector used `sorted(set(ids))`. This is not an integrity failure or confidence ranking. The original response order is preserved in the saved payload.

**All seven ambiguous queries:** Adenosylcobalamin (11 CIDs), L-5-Methyltetrahydrofolate calcium (3), Lipoic Acid (3), Magnesium Taurinate (4), Methylcobalamin (25), Vitamin D2 (2), Vitamin K2 (3).

**Important examples:** Magnesium and Magnesium Citrate return different CIDs (5462224 / 6099959). Creatine, Creatine Monohydrate and Creatine HCl return 586 / 80116 / 134732. Ashwagandha and Turmeric return no candidates for those saved name searches. These observations are not approvals of ingredient identity, synonymy or parent/form equivalence.

**Quality boundaries:** category membership comes from DSLD search provenance and includes co-ingredients, so it must not become an indication or compound-class assertion. PubChem identifiers/properties support chemical representation, not label dosage, safety or clinical efficacy. Retrieval dates are not chemical-record revision dates. Botanical mixtures, organisms and broad supplement categories must not be forced into a single compound mapping.

**Schema implication:** separate `Ingredient`, `ChemicalCompound` (namespaced CID) and a provenance-backed `MappingCandidate`. Preserve query text/variants, source occurrences, method, candidate set, timestamp and review state. Neither one CID returned nor a shared synonym should produce an automatic identity merge.

The cells below show the complete summary, property completeness and real source examples. Full query, request, occurrence and compound data remain available in `pubchem_source`; full audit details are in `pubchem_audit_report`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

pubchem_root = Path.cwd()
if not (pubchem_root / 'data/raw/pubchem').is_dir():
    pubchem_root = pubchem_root.parent
pubchem_run = subprocess.run(
    [sys.executable, str(pubchem_root / 'scripts/audit_pubchem.py')],
    cwd=pubchem_root, check=True, capture_output=True, text=True,
)
pubchem_audit_dir = Path(pubchem_run.stdout.splitlines()[0])
pubchem_audit_report = json.loads((pubchem_audit_dir / 'audit.json').read_text())
pubchem_field_profiles = json.loads((pubchem_audit_dir / 'field_profiles.json').read_text())
pubchem_source = json.loads((pubchem_root / 'data/raw/pubchem/runs/20261028T035149634190Z/collection.json').read_text())
print('Report:', pubchem_audit_dir.relative_to(pubchem_root))
print(json.dumps(pubchem_audit_report['summary'], indent=2))
print('Property completeness:')
print(json.dumps(pubchem_field_profiles['properties'], indent=2))


Report: data/processed/pubchem/pubchem-baseline-audit-v1-ee86973dbcbcc97b
{
  "source_labels": 400,
  "query_groups": 739,
  "original_name_variants": 753,
  "dsld_occurrences_verified": 3391,
  "occurrence_kinds": {
    "form": 1052,
    "ingredient": 2339
  },
  "search_status": {
    "ok": 228,
    "not_found": 511
  },
  "queries_with_response_order_changed_by_collector": 6,
  "single_candidate_queries": 221,
  "ambiguous_queries": 7,
  "compound_records": 239,
  "candidate_edges": 272,
  "cids_reached_by_multiple_queries": 30,
  "synonym_entries": 46426,
  "exact_duplicate_synonym_entries_within_cid": 0,
  "normalized_synonyms_shared_across_cids": 74,
  "property_collision_groups": {
    "InChIKey": 8,
    "InChI": 8,
    "ConnectivitySMILES": 13,
    "MolecularFormula": 21,
    "Title": 0
  },
  "missing_iupac_names": [
    {
      "cid": "10475633",
      "title": "Arterolane"
    },
    {
      "cid": "92003852",
      "title": "Boron glycinate"
    },
    {
      "cid": "16659

In [ ]:
for key in ['magnesium', 'magnesium citrate', 'creatine', 'creatine monohydrate', 'creatine hcl', 'ashwagandha', 'turmeric']:
    query = pubchem_source['queries'][key]
    result = pubchem_source['search_results'][key]
    print('\nQUERY:', query['query'])
    print(json.dumps({'variants':query['variants'], 'example_dsld_occurrence':query['occurrences'][0],
                      'saved_response':result['response'],
                      'candidate_properties':[pubchem_source['records'][str(cid)]['properties'] for cid in result['cids']]},
                     ensure_ascii=False, indent=2))
print('\nALL AMBIGUOUS QUERIES (CID/title pairs; no selection approved)')
for row in pubchem_audit_report['mapping_review']:
    if row['candidate_count'] > 1:
        print(json.dumps({'query':row['query'], 'candidates':row['candidates']}, ensure_ascii=False, indent=2))
print('\nSHARED-CID EXAMPLES')
print(json.dumps(pubchem_audit_report['shared_cids'][:5], ensure_ascii=False, indent=2))
print('\nSHARED INCHIKEY EXAMPLES — DISTINCT CIDS RETAINED')
print(json.dumps(pubchem_audit_report['property_collisions']['InChIKey'][:3], indent=2))
print('\nTITLE/SYNONYM TEXT-MATCH REVIEW EXAMPLES')
for row in pubchem_audit_report['mapping_review']:
    if any(not c['query_matches_title_or_synonym'] for c in row['candidates']):
        print(json.dumps({'query':row['query'], 'candidates':row['candidates']}, indent=2))



QUERY: Magnesium
{
  "variants": [
    "Magnesium"
  ],
  "example_dsld_occurrence": {
    "dsld_id": "19279",
    "search_categories": [
      "Magnesium"
    ],
    "ingredient_id": 69771,
    "name": "Magnesium",
    "kind": "ingredient",
    "json_path": "ingredientRows[0]"
  },
  "saved_response": {
    "IdentifierList": {
      "CID": [
        5462224
      ]
    }
  },
  "candidate_properties": [
    {
      "CID": 5462224,
      "MolecularFormula": "Mg",
      "MolecularWeight": "24.305",
      "ConnectivitySMILES": "[Mg]",
      "InChI": "InChI=1S/Mg",
      "InChIKey": "FYYHWMGAXLPEAU-UHFFFAOYSA-N",
      "IUPACName": "magnesium",
      "Title": "Magnesium"
    }
  ]
}

QUERY: Magnesium Citrate
{
  "variants": [
    "Magnesium Citrate"
  ],
  "example_dsld_occurrence": {
    "dsld_id": "19279",
    "search_categories": [
      "Magnesium"
    ],
    "ingredient_id": 63832,
    "name": "Magnesium Citrate",
    "kind": "form",
    "json_path": "ingredientRows[0].forms[1]"
  }

### 15.1 PubChem POC field selection — provisional

**Purpose:** identify chemical candidates for DSLD ingredient names and forms, support terminology lookup, and preserve the evidence needed to review a mapping. This is a selection of existing fields, not the final shared schema, an approved ingredient–compound equivalence or a transformation of the raw dataset.

#### Selected fields

| Category | Existing source fields to retain | Use / boundary |
|---|---|---|
| Compound identity | `records` key; `records[cid].cid`; `properties.CID`, `properties.Title`, `properties.IUPACName` | Namespace compound identities as PubChem CIDs. Preserve missing IUPAC names; do not invent a replacement or use a title as the identity key. |
| Chemical representation | `properties.MolecularFormula`, `MolecularWeight`, `ConnectivitySMILES`, `InChI`, `InChIKey` | Retain source representations for chemical review, with original value types. Shared formula, connectivity or InChIKey is not an automatic ingredient/formulation merge. Molecular weight is not a dose. |
| Search aliases | `synonyms.CID`, `synonyms.Synonym` | Preserve source synonym membership for search/review. Aliases are not unique identifiers, validated clinical claims or a universal merge dictionary. Display a relevant subset without discarding the underlying source list. |
| Originating query | `queries` key; `query`, `variants` | Retain the submitted name and original variants. A broad supplement topic is not necessarily a single compound. |
| DSLD ingredient identity/context | `queries[key].occurrences`: `dsld_id`, `ingredient_id`, `name`, `kind`, `json_path`, `search_categories` | Preserve each ingredient/form occurrence and its source path. Category provenance includes co-ingredients and is not a clinical indication. |
| Mapping candidates | `search_results[key].status`, `cids`, `response` | Keep the full returned candidate set, including no-match outcomes. One CID returned does not approve identity; preserve salts/forms and ambiguity. |
| Request provenance | `search_results[key].requests`: URL, request timestamp, attempt, HTTP status and response | Preserve the actual name-lookup method and original response. Derived CID lists were sorted; order is not a confidence ranking. |
| Compound provenance | `compound_responses`: `kind`, `requested_cids`, status, response and request traces | Link selected properties/synonyms to the corresponding saved batch response; do not create a separate clinical source from a duplicated response copy. |
| Dataset provenance | `source`, `scope`, `started_at_utc`, `finished_at_utc`, `dsld_provenance`; collection/manifest paths and hashes | Bind mappings to the source-label snapshot and chemical snapshot. Retrieval time is not a compound revision date. |
| Review metadata (derived, not a raw PubChem field) | Candidate-review state, ambiguity/no-match reason, source-validation result, audit-rule version | Successful searches remain `unreviewed`; empty results remain `no_candidate_returned`. Do not invent a reviewer, confidence score or approval. |

#### Clinical-field boundary

The NatMed clinical selection does **not** transfer directly to PubChem. This saved collection does not contain source-supported clinical claim types, indications/populations, effectiveness/safety ratings, dose regimens, interaction partners or study citations. Mark these as **not supplied by this dataset**, rather than interpreting missing fields as no risk, no interaction or no evidence. Do not populate them from compound names, synonyms or DSLD search categories.

For evidence attribution, retain the original DSLD occurrence and PubChem API response/value, not an invented prose excerpt. Source URLs and response hashes provide provenance; they are not clinical citations. Future links to clinical assertions must come from their own sources and reviewed ingredient mappings.

#### Deferred from the first graph projection

- Transport throttle diagnostics and routine retry details as graph entities: retain them in raw request logs and keep access via provenance.
- Bulk synonym nodes and guessed identifiers parsed from synonym strings: keep aliases as source-scoped metadata until a concrete retrieval/mapping need justifies expansion.
- Uncollected chemical properties, pharmacology, efficacy or safety information: do not assume the saved CID snapshot includes them.

#### Preservation and example rules

- Keep absent/null fields and failed/no-match searches distinguishable. No-match does not establish absence under other names or lookup strategies.
- Reuse a compound record by CID while preserving every incoming mapping candidate and evidence link. Do not merge DSLD ingredient identities merely because queries share a CID.
- Magnesium / Magnesium Citrate retain CIDs 5462224 / 6099959; Creatine / Creatine Monohydrate / Creatine HCl retain 586 / 80116 / 134732. These are saved candidates, not mapping approvals.
- Ashwagandha and Turmeric name searches returned no candidates in this snapshot; keep their ingredient identities even without a chemical mapping.

**Selection only:** no mappings approved, data normalized or records merged. Finalize the shared schema after all dataset selections. `read.md` remains unchanged.


## 16. RxNorm baseline audit — shared audit categories

**Scope:** offline audit of the saved 2026-10-28 snapshot. No fresh queries, identity approvals, corrections or merging.

| Category | Findings | Interpretation |
|---|---|---|
| Structure and completeness | 739 query groups / 753 source-name variants; 287 concept records with properties and allProperties responses | Concept snapshots and query results are separate entities; properties are not label doses |
| Source links | All 3,391 DSLD occurrences verified across 400 labels (including forms and nested ingredients) | Other ingredients are outside this collection scope |
| Search outcomes | 282 exact matches, 31 normalized matches, 426 no matches | Search method does not establish chemical equivalence |
| Duplicates and ambiguity | 31 ambiguous queries: 21 exact and 10 normalized; 282 single-candidate queries | Exact-search count and single-candidate count happen to be equal but describe different subsets |
| Identifier reuse | 347 candidate links; 46 RxCUIs reached by multiple query names | Reuse concept nodes; keep each ingredient mapping unreviewed |
| Concept distinctions | 256 IN, 30 PIN, 1 MIN; all source suppression flags are N | Preserve term type rather than collapsing all records into one ingredient type |
| Completeness | 226 blank basic synonym fields; all 287 basic umlscui fields blank | These do not mean the concepts lack names or other name metadata; this collection does not supply UMLS CUIs |
| Property volume | 180,747 property entries, including 175,552 SPL_SET_ID entries | Large file size mainly reflects source-code volume, not many additional ingredients or independent evidence |
| Shared external codes | 31 shared groups excluding SPL_SET_ID: 15 UNII_CODE, 14 DRUGBANK, 1 MMSL_CODE, 1 SNOMEDCT | External-code overlap is review evidence, not permission to merge RxCUIs |
| Provenance | Vocabulary 05-Oct-2026 and API 3.1.356 at start and end; integrity checks pass | Stable captured version; no live currency or chemical correctness verification |

**Mapping quality examples:**

- `Magnesium` maps by exact search to 6574 (`magnesium`); `Magnesium Citrate` to 52356 (`magnesium citrate`). Preserve the distinction.
- `Creatine` maps to 2907; `Creatine Monohydrate` to 1310467. `Creatine HCl` returns **both** via normalized search. Do not treat this as verification of the hydrochloride form.
- `Turmeric` returns 1114883 (`turmeric extract`) through exact search. The term scope still needs review: the search-method label does not mean identical wording or botanical preparation.
- `Ashwagandha` returns no match under the saved exact-then-normalized policy. This does not prove absence under all possible names.

There are 193 query/candidate preferred-name text mismatches. This is a conservative text flag, not a count of wrong mappings; aliases and source naming conventions can differ. All matched mappings retain their review requirement. No approximate searches were collected.

**Schema implication:** retain a namespaced `RxNormConcept`, its term type and source version, and separate provenance-backed `MappingCandidate` assertions. Preserve source query, DSLD occurrence, exact/normalized method, full candidate set and review state. Category provenance and prescribing-related attributes must not become efficacy, safety or patient-specific recommendations.

The cells below show saved outputs, field completeness, the example source searches and all ambiguous/normalized candidates. Full raw responses remain accessible in `rxnorm_source`; detailed findings are in `rxnorm_audit_report`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

rxnorm_root = Path.cwd()
if not (rxnorm_root / 'data/raw/rxnorm').is_dir():
    rxnorm_root = rxnorm_root.parent
rxnorm_run = subprocess.run(
    [sys.executable, str(rxnorm_root / 'scripts/audit_rxnorm.py')],
    cwd=rxnorm_root, check=True, capture_output=True, text=True,
)
rxnorm_audit_dir = Path(rxnorm_run.stdout.splitlines()[0])
rxnorm_audit_report = json.loads((rxnorm_audit_dir / 'audit.json').read_text())
rxnorm_field_profiles = json.loads((rxnorm_audit_dir / 'field_profiles.json').read_text())
rxnorm_source = json.loads((rxnorm_root / 'data/raw/rxnorm/runs/20261028T060819025094Z/collection.json').read_text())
print('Report:', rxnorm_audit_dir.relative_to(rxnorm_root))
print(json.dumps(rxnorm_audit_report['summary'], indent=2))
print('Basic concept-property completeness:')
print(json.dumps(rxnorm_field_profiles['concept_properties'], indent=2))


Report: data/processed/rxnorm/rxnorm-baseline-audit-v1-bb5c0ce9a6faee14
{
  "source_labels": 400,
  "query_groups": 739,
  "name_variants": 753,
  "dsld_occurrences_verified": 3391,
  "search_status": {
    "matched": 313,
    "not_found": 426
  },
  "match_methods": {
    "exact": 282,
    "normalized": 31
  },
  "single_candidate_queries": 282,
  "ambiguous_queries": 31,
  "ambiguity_by_method": {
    "exact": 21,
    "normalized": 10
  },
  "concept_records": 287,
  "candidate_links": 347,
  "term_types": {
    "IN": 256,
    "PIN": 30,
    "MIN": 1
  },
  "suppression_flags": {
    "N": 287
  },
  "rxcuis_reached_by_multiple_queries": 46,
  "shared_normalized_property_names": 0,
  "shared_external_code_groups_excluding_spl_set_id": 31,
  "shared_code_groups_by_system": {
    "MMSL_CODE": 1,
    "DRUGBANK": 14,
    "UNII_CODE": 15,
    "SNOMEDCT": 1
  },
  "property_entries": 180747,
  "property_categories": {
    "ATTRIBUTES": 989,
    "CODES": 177517,
    "NAMES": 640,
    "SOURCE

In [ ]:
for key in ['magnesium', 'magnesium citrate', 'creatine', 'creatine monohydrate', 'creatine hcl', 'turmeric', 'ashwagandha']:
    query = rxnorm_source['queries'][key]
    result = rxnorm_source['search_results'][key]
    print('\nQUERY:', query['query'])
    print(json.dumps({'example_dsld_occurrence':query['occurrences'][0],
        'saved_searches':[{'mode':s['mode'],'status':s['status'],'response':s['response'],
                          'request_parameters':[r['params'] for r in s['requests']]} for s in result['searches']],
        'match_method':result['match_method'], 'requires_identity_review':result['requires_identity_review'],
        'candidate_properties':[rxnorm_source['records'][rid]['properties']['response']['properties'] for rid in result['rxcuis']]},
        ensure_ascii=False, indent=2))
print('\nALL AMBIGUOUS OR NORMALIZED MATCHES — UNAPPROVED')
for row in rxnorm_audit_report['mapping_review']:
    if row['candidate_count'] > 1 or row['match_method'] == 'normalized':
        print(json.dumps({'query':row['query'],'method':row['match_method'],'candidates':row['candidates']}, ensure_ascii=False, indent=2))
print('\nSHARED CONCEPT EXAMPLES')
print(json.dumps(rxnorm_audit_report['shared_concepts'][:5], ensure_ascii=False, indent=2))
print('\nSHARED EXTERNAL CODE EXAMPLES — DISTINCT CONCEPTS RETAINED')
print(json.dumps(rxnorm_audit_report['shared_external_codes'][:5], ensure_ascii=False, indent=2))



QUERY: Magnesium
{
  "example_dsld_occurrence": {
    "dsld_id": "19279",
    "search_categories": [
      "Magnesium"
    ],
    "ingredient_id": 69771,
    "name": "Magnesium",
    "kind": "ingredient",
    "json_path": "ingredientRows[0]"
  },
  "saved_searches": [
    {
      "mode": "exact",
      "status": "ok",
      "response": {
        "idGroup": {
          "rxnormId": [
            "6574"
          ]
        }
      },
      "request_parameters": [
        {
          "name": "Magnesium",
          "allsrc": 0,
          "search": 0
        }
      ]
    }
  ],
  "match_method": "exact",
  "requires_identity_review": true,
  "candidate_properties": [
    {
      "rxcui": "6574",
      "name": "magnesium",
      "synonym": "",
      "tty": "IN",
      "language": "ENG",
      "suppress": "N",
      "umlscui": ""
    }
  ]
}

QUERY: Magnesium Citrate
{
  "example_dsld_occurrence": {
    "dsld_id": "19279",
    "search_categories": [
      "Magnesium"
    ],
    "ingredient_i

### 16.1 RxNorm POC field selection — provisional

**Purpose:** represent terminology concepts and reviewable links from DSLD ingredient names/forms. Preserve concept specificity and search method. This is not the finalized shared schema or approval of a chemical, product or clinical equivalence.

#### Selected fields

| Category | Existing source fields to retain | Use / boundary |
|---|---|---|
| Concept identity | `records` key; `records[rxcui].rxcui`; `properties.response.properties.rxcui`, `name` | Namespace RxCUIs. Preferred name is display metadata, not a merge key. |
| Concept type/status | `properties.response.properties.tty`, `language`, `suppress` | Keep term-type distinctions (IN/PIN/MIN) and source status explicit. A non-suppressed concept is not an approved ingredient mapping or a safety statement. |
| Alternate terminology names | Basic `synonym`; `all_properties.response.propConceptGroup.propConcept` entries with `propCategory=NAMES` | Retain source-scoped names for lookup. Blank basic synonym does not mean no other name metadata. |
| External mapping candidates | Selected `propCategory=CODES` entries: `propName`, `propValue`, initially `UNII_CODE`, `DRUGBANK`, `SNOMEDCT` | Preserve the code namespace and all returned values. These are source cross-references, not verified one-to-one equivalences; no automatic cross-source merging. |
| Source vocabulary metadata | `propCategory=SOURCES` entries, keeping category/name/value together | Attribute terminology metadata to its vocabulary sources. This is not a bibliography of studies. |
| Originating query | `queries` key; `query`, `variants` | Preserve actual submitted text and variants; form-qualified queries must not silently become parent-ingredient mappings. |
| DSLD occurrence context | `queries[key].occurrences`: `dsld_id`, `ingredient_id`, `name`, `kind`, `json_path`, `search_categories` | Retain ingredient/form role, source path and retrieval context. Categories are not indications. |
| Candidate result and review flag | `search_results[key].status`, `match_method`, `rxcuis`, `requires_identity_review` | Preserve every candidate and exact/normalized method. `requires_identity_review=False` on a no-match record does not mean an identity was approved. |
| Search provenance | `searches`: mode, status, response, requests including URL, params, timestamp, attempt and HTTP status | Keep exact-search results and fallback normalized results; no approximate search was collected. “Exact” is a search-method label, not a unique-identity guarantee. |
| Concept-response provenance | `properties` and `all_properties`: status, `valid`, response and requests | Keep response links/hashes so selected fields can be reconstructed. Upstream `valid` indicates collection validation, not clinical validity. |
| Dataset/version provenance | `source`, `scope`, `search_policy`, `started_at_utc`, `finished_at_utc`, `version_start`, `version_end`, `dsld_provenance`; manifest paths/hashes | Bind each concept/mapping to the captured vocabulary/API versions and source labels. |
| Local review metadata (derived) | Mapping-review state, ambiguity/no-match reason, validation results and audit-rule version | Matched concepts remain `unreviewed`; preserve missing/ambiguous information without invented confidence or clinical approval. |

#### Clinical-field boundary

This saved RxNorm collection does not supply validated clinical claim types, indications/populations, effectiveness/safety ratings, recommended dose context, interaction partners or study citations. They are **not supplied**, not clinical negatives. Do not infer them from terminology names, `PRESCRIBABLE`, `RXNAV_HUMAN_DRUG`, codes or category associations. Label dose comes from DSLD; clinical context comes from separately sourced annotations such as NatMed, after appropriate mapping and review.

Original supporting evidence here means the DSLD occurrence plus the exact search/concept API response—not a fabricated clinical excerpt. Keep API URLs and vocabulary source names for attribution, without presenting them as studies supporting treatment claims.

#### Deferred from the first graph projection

- Bulk `SPL_SET_ID` links (175,552 entries in this snapshot): preserve in raw allProperties, but do not expand into the initial graph without a specific product-label integration need.
- Other code systems and attributes not required for ingredient identity review: retain in raw responses for later scoped selection.
- `umlscui` as a usable identifier: all 287 basic values are blank. Preserve that missingness in source/audit metadata; do not invent UMLS mappings.
- Clinical interpretations of source attributes or vocabulary relationships not actually collected.

#### Preservation and example rules

- Preserve missing/null/empty values and search outcomes separately. A no-match query remains an ingredient with an unresolved terminology link.
- Reuse concept nodes by RxCUI; retain distinct query/occurrence mappings, term types and review state. Shared external codes do not justify merging concepts.
- `Creatine HCl` returned creatine (2907) and creatine monohydrate (1310467) via normalized search. Keep both as unresolved candidates, not hydrochloride equivalences.
- `Turmeric` returned `turmeric extract` (1114883) through exact search. Preparation/scope review remains necessary despite that method label.
- Preserve original source identifiers and concept versions for later incremental updates; do not overwrite a past mapping decision using a newer result without review history.

**Selection only:** no concepts merged or mappings approved. Finalize the shared schema after selections for all datasets. `read.md` remains unchanged.


## 17. NatMed baseline audit and clinical-annotation schema/source validation

**Scope:** the first two NatMed steps only: baseline audit and validation of the existing annotations. This is not clinical approval, new extraction, normalization, deduplication or a replacement schema. The saved upstream snapshot is inspected offline.

| Audit category | Verified findings / limitations |
|---|---|
| Structure and completeness | 10 monographs, 10 HTML captures, 1,179 passages and 1,269 pending claims. Field profiles and claim-type completeness are generated separately. |
| Identifiers | Record, passage and claim IDs are unique; graph endpoints resolve. Source references, section positions and source-title ingredient identities are checked. |
| Duplicates and ambiguity | 5 repeated passage-text groups, 13 repeated claim-text groups and 12 shared source-span/type groups. **Zero identical full claim payloads after excluding only ID.** Shared text may support different populations/doses, so no merging is performed. |
| Mapping quality | Exact monograph title is required for claim ingredient identity. Fish Oil is not all Omega-3; Lactobacillus acidophilus is not all probiotics. Cross-source ingredient mappings remain unapproved. |
| Provenance | Local manifest hashes, upstream canonical input/schema fingerprints, source-text and HTML hashes, reference-document hashes and passage word windows pass. No live publisher comparison is performed. |

### Schema and source-support results

- All **1,269 claims** pass the supplied **claim contract 1.1** JSON Schema using **Draft 2020-12**, including URI/date-time format checks.
- Claim excerpts match their source section offsets; source hashes, URLs, headings and exact source-title ingredient identities pass.
- All **2,923 field-support anchors** pass the implemented bounds/value checks, with support keys required for the non-null annotations specified by the contract. Numeric checks permit decimal representation differences, not unit conversion.
- **769 effectiveness-rating anchors precede the claim excerpt**, as the contract permits for an earlier group heading. The audit checks intervening standalone ratings known from the annotations; this is not exhaustive validation of the publisher's HTML grouping.
- Dose minimum/maximum ordering passes. No claim review state is changed: **1,269 pending, zero approved**.
- The upstream graph remains **20 nodes / 10 MENTIONS edges**, not a clinical claim graph.

### Unresolved findings and interpretation

The upstream extraction report records **49 issues** and **55 inline references absent from its captured bibliographies**. These are preserved upstream findings, not independently established publisher errors. Original-page accuracy, citation-to-claim support and clinical relevance remain unverified.

All 85 dose claims lack a structured indication; formulation and strain are null for those dose claims. Only 71 have a structured numeric amount/range and unit. Preserve unknowns and source text rather than guessing missing context. Detailed completeness varies by claim type and is shown below.

An important duplicate example: several Vitamin B12 annotations share a source paragraph but describe different populations and amounts. Equal text/source spans do not make those annotations interchangeable. The example cell displays the differing population and dose fields without interpreting them as recommendations.

**Validation boundary:** this notebook runs an independent local audit and the supplied JSON Schema; it does not execute the upstream custom Python validator. Source-span validity does not certify a correct medical association or complete extraction. The original-source and clinical-review flags remain unresolved.

**Dependency:** `supplementsrx_env/bin/python -m pip install -r requirements-audit.txt` installs the JSON Schema validator. The cells use the project environment when available. Existing raw files remain unchanged.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

natmed_root = Path.cwd()
if not (natmed_root / 'data/raw/natmedpro').is_dir():
    natmed_root = natmed_root.parent
natmed_python = natmed_root / 'supplementsrx_env/bin/python'
if not natmed_python.exists():
    natmed_python = Path(sys.executable)
natmed_run = subprocess.run(
    [str(natmed_python), str(natmed_root / 'scripts/audit_natmed.py')],
    cwd=natmed_root, check=True, capture_output=True, text=True,
)
natmed_audit_dir = Path(natmed_run.stdout.splitlines()[0])
natmed_audit_report = json.loads((natmed_audit_dir / 'audit.json').read_text())
natmed_field_profiles = json.loads((natmed_audit_dir / 'field_profiles.json').read_text())
natmed_base = natmed_root / 'data/raw/natmedpro'
natmed_records = {r['id']:r for r in map(json.loads,(natmed_base / 'records.jsonl').read_text().splitlines())}
natmed_claims = {c['id']:c for c in map(json.loads,(natmed_base / 'upstream_derived/pending_clinical/clinical_claims.jsonl').read_text().splitlines())}
print('Report:', natmed_audit_dir.relative_to(natmed_root))
print(json.dumps(natmed_audit_report['summary'], indent=2))
print('CLAIM-TYPE COMPLETENESS')
print(json.dumps(natmed_audit_report['field_coverage_by_claim_type'], indent=2))


Report: data/processed/natmedpro/natmed-baseline-audit-v1-b548c344b8c0a9b7
{
  "monographs": 10,
  "passages": 1179,
  "claims": 1269,
  "html_captures": 10,
  "claim_types": {
    "safety": 107,
    "adverse_effect": 120,
    "effectiveness": 809,
    "dose": 85,
    "interaction": 148
  },
  "review_states": {
    "pending": 1269
  },
  "graph_nodes": 20,
  "graph_edges": 10,
  "graph_edge_types": {
    "MENTIONS": 10
  },
  "schema_draft": "2020-12",
  "claim_contract": "1.1",
  "jsonschema_version": "4.26.0",
  "schema_valid_claims": 1269,
  "schema_error_count": 0,
  "source_support_error_count": 0,
  "field_support_anchors_checked": 2923,
  "earlier_effectiveness_heading_anchors": 769,
  "integrity_error_count": 0,
  "duplicate_groups": {
    "record_text": 0,
    "passage_text": 5,
    "claim_text": 13,
    "claim_payload_except_id": 0,
    "claim_source_span_and_type": 12
  },
  "monographs_requiring_review": 10,
  "upstream_reported_extraction_issues": {
    "section_not_in_ht

In [ ]:
print('SOURCE IDENTITY / COVERAGE')
print(json.dumps(natmed_audit_report['sources'], ensure_ascii=False, indent=2))
print('ONE PENDING CLAIM PER TYPE — SOURCE TEXT, NOT RECOMMENDATIONS')
example_claims = {}
for claim in natmed_claims.values():
    example_claims.setdefault(claim['claim_type'], claim)
for kind, claim in sorted(example_claims.items()):
    print('\n', kind)
    print(json.dumps(claim, ensure_ascii=False, indent=2))
print('SHARED SOURCE-SPAN EXAMPLE — DIFFERENT ANNOTATIONS')
group = natmed_audit_report['duplicates']['claim_source_span_and_type'][0]
for claim_id in group:
    claim = natmed_claims[claim_id]
    print(json.dumps({k:claim[k] for k in ['id','claim_type','ingredient','population','dose','review']}, ensure_ascii=False, indent=2))
print('FIELD-SUPPORT EXAMPLE VERIFIED AGAINST THE SOURCE SECTION')
claim = next(iter(natmed_claims.values()))
section = natmed_records[claim['source']['record_id']]['sections'][claim['source']['section_index']]['text']
for field, anchor in claim['field_support'].items():
    print(json.dumps({'claim_id':claim['id'],'field':field,'anchor':anchor,
                      'source_text':section[anchor['start']:anchor['end']]}, ensure_ascii=False, indent=2))


SOURCE IDENTITY / COVERAGE
[
  {
    "id": "natmed:0d296c0daa1cccb456da568c",
    "title": "Ashwagandha",
    "topics": [
      {
        "supplement": "Ashwagandha",
        "matched_terms": [],
        "mapping_method": "verified_exact_source_title"
      }
    ],
    "sections": 18,
    "passages": 51,
    "claims": 73,
    "source_updated_at": "10/28/2026",
    "collected_at": "2026-10-28T23:25:05.335Z",
    "requires_clinical_review": true,
    "capture_complete": false,
    "bibliography_coverage": {
      "bibliography_entries": 137,
      "inline_citation_refs": 119,
      "inline_citation_refs_missing_from_bibliography": 0,
      "missing_sample": []
    }
  },
  {
    "id": "natmed:0e9f2262886c394a16e6a7b6",
    "title": "Vitamin B12",
    "topics": [
      {
        "supplement": "Vitamin B12",
        "matched_terms": [],
        "mapping_method": "verified_exact_source_title"
      }
    ],
    "sections": 18,
    "passages": 77,
    "claims": 96,
    "source_updated_at": 

## 18. NatMed Pro POC field selection — provisional, before shared-schema finalization

**Decision scope:** select which existing NatMed fields the POC needs. This is not a finalized graph schema, a new extraction, an identity mapping approval, or clinical approval. Finalize the shared schema only after field selection across all datasets. Keep original files and upstream-derived artifacts unchanged; defer fields from the initial projection, not from storage.

### User needs supported

- Purpose and effectiveness: source-reported indication, effectiveness rating, population and qualifying text.
- Safety and interactions: source-reported safety statements, adverse effects, interaction partners and qualifications.
- Amount and usage context: source-reported dose, preparation, population, regimen and bounds; distinguish these from DSLD label directions and from personalized recommendations.
- Evidence transparency: exact excerpts, source identity, citations, dates, limitations and review status.
- Product comparison: NatMed provides ingredient-level reference context, not proof that a particular DSLD product has the same formulation or effects.

### A. Selected clinical-annotation fields

Source: `data/raw/natmedpro/upstream_derived/pending_clinical/clinical_claims.jsonl`.

| Category | Keep these existing fields | Interpretation / preservation rule |
|---|---|---|
| Annotation identity and type | `id`, `claim_type` | Retain all five types: effectiveness, safety, dose, adverse_effect and interaction. Do not conflate claims because text overlaps. |
| Source-specific ingredient identity | `ingredient.name`, `ingredient.formulation`, `ingredient.strain` | Keep the exact monograph identity and explicit preparation/strain. Null stays unknown. Fish Oil is not all Omega-3; Lactobacillus acidophilus is not all probiotics. |
| Clinical context | `population`, `indication` | Preserve intended population and use where explicitly captured. An absent indication is not a general-use recommendation. |
| Effectiveness | `effectiveness_rating` | Keep the publisher wording, including negative and insufficient-evidence categories. Do not translate into a new numerical score. |
| Safety / adverse effects | `safety_rating`, `adverse_effect_category` | Interpret with population and full excerpt. Do not infer no risk from absent fields. |
| Dose and regimen | All of `dose`: `text`, `value`, `minimum`, `maximum`, `unit`, `qualifier`, `regimen`, `frequency`, `duration.value`, `duration.unit`, `duration.qualifier`, `route` | Keep bounds, alternatives and context. Preserve composite units such as mg/kg/day. A source amount is not automatically a recommendation. Do not normalize or fill missing fields at selection time. |
| Interactions | All of `interaction`: `partner`, `rating`, `severity`, `occurrence` | Keep these dimensions separate; do not substitute rating for severity or convert occurrence wording into an invented probability. |
| Evidence classification | `evidence_grade` | Retain when present. Do not infer it from effectiveness rating, study count or publication type. |
| Full statement context | `claim_text` | Keep the verbatim source excerpt, including caveats. It supports interpretation of structured values and text-only unresolved entries. |

**Not every selected field is populated.** All 85 dose annotations currently lack a structured indication, formulation and strain; 71 have numeric amounts/ranges with units. Preserve text-only annotations and unknowns. Do not delete records solely because a selected field is null.

### B. Required internal provenance and review fields

These fields need not appear in every short answer, but must remain attached to the selected annotations.

| Source | Fields to keep | Why |
|---|---|---|
| Clinical annotation | All of `source`: `record_id`, `url`, `content_hash`, `section_index`, `section_heading`, `start`, `end`, `excerpt` | Reconstruct exactly which source version and span supports the annotation. Preserve section meaning: a condition mentioned in a caution is not necessarily an indication. |
| Clinical annotation | `field_support` | Retain the support offsets for each structured non-null annotation. An earlier effectiveness heading may be outside the excerpt under the existing contract. |
| Clinical annotation | All of `review`: `state`, `reviewer`, `reviewed_at` | Preserve pending/approved/rejected state. All 1,269 current annotations remain pending. |
| Source monograph | `id`, `title`, `source`, `record_type`, `evidence_role`, `source_url`, `content_hash`, `collected_at`, `source_updated_at`, `source_batch_id`, `schema_version` | Distinguish reference source, capture/version, and retrieval dates. Page-update dates are not study publication dates or literature-review cutoffs. |
| Source monograph | `requires_clinical_review`, `capture_quality`, `normalization`, `supplements` | Retain capture limitations and topic-mapping context. Broad topic membership is not an approved ingredient equivalence. |
| Upstream manifests / audit outputs | Input/schema fingerprints, extraction method, claim contract version, issue codes and affected source/claim references where available | Make extraction limitations and audit findings traceable. Link existing artifacts rather than inventing a raw claim field. |

### C. Source text and retrieval fields to retain

Source monographs remain the authority for the captured text. Keep `sections` (`heading`, `text`) and access to full `text` for resolving offsets and qualifications. For the initial retrieval view, prioritize safety, warnings, adverse effects, effectiveness, dosing, formulation and interaction/contraindication sections, **using actual saved headings** rather than assuming derived excerpt keys equal headings.

Some warning/context text is not represented in the extracted claims. Keep access to the original monograph and relevant context (including overview when needed), and do not treat an absent annotation as absent source information.

For existing `passages.jsonl`, select:

- Identity/text: `id`, `record_id`, `title`, `section`, `text`.
- Source binding: `source`, `record_type`, `evidence_role`, `source_url`, `source_content_hash`, `section_index`, `word_start`, `word_count`, `chunk_words`, `source_batch_id`.
- Citation context: `citation_links`, `citation_scope`, `reference_scope`, `reference_documents_sha256`.
- Status/context: `collected_at`, `source_updated_at`, `requires_clinical_review`, `clinical_review`, `capture_quality`, `supplements`, `normalization`.

Do not count overlapping passages as independent evidence. Do not discard an overlapping passage solely on text similarity before preserving its source and citation context.

### D. Citation and reference selection

- Retain monograph `citation_links` (`label`, `url`) and `reference_documents` (`url`, `title`, `text`, `captured_at`, `content_hash`) as source evidence.
- Retain original `references` text as supporting source material, but do not assume it is a clean bibliography: some values contain footer or literature-review text. The captured reference documents and scoped citation links need separate treatment.
- Clinical claims do not currently have a dedicated structured citation list. Later linking must use their source spans and explicitly cited reference labels. Do not attach every monograph reference or all overlapping-passage citations to every claim.
- Existing PubMed metadata may enrich identified citations later. Do not ingest all 5,990 PMID records as if each independently supports each claim; first establish citation-specific links. This selection does not validate study relevance or full-text support.
- Preserve missing-reference flags. Citation presence does not certify that a study supports the annotated association.

### E. Deferred from the first POC projection

Retain these in raw/imported storage, but do not initially expand them into answer-facing graph facts:

- Full HTML bodies and capture-control diagnostics: keep their hashes/links and retain the artifacts for source verification.
- Exhaustive mechanisms, pharmacokinetics and classification content: available in source sections, but outside the initial structured claim projection unless needed to qualify a selected claim.
- Unscoped bibliography-to-claim relationships and bulk PubMed enrichment without identified supporting citations.
- The upstream Source/Supplement `MENTIONS` graph as clinical evidence. Its source/topic links can be referenced, but are not effectiveness or safety assertions.
- Reddit-only fields on shared passage records, such as `thread_id`, `parent_id` and `mention_scope`, when null for NatMed. Those fields belong in the later Reddit selection.

### Selection boundaries and next step

The selected fields preserve the five user needs while keeping medical statements distinct from product-label information and terminology mappings. They do not authorize normalization of medical meaning, broadening ingredient scope, changing review states, correcting doses or merging annotations.

**Next:** complete the selection sets for the remaining datasets, compare shared identities/provenance requirements, and only then finalize the combined schema and transformation rules. `read.md` is intentionally unchanged at this selection stage.


## 19. Reddit baseline audit and structural/source-support validation

**Scope:** all 4,880 Reddit source records (287 posts, 4,593 comments) and 5,071 upstream passages in the imported snapshot. The same five audit categories are used, but Reddit has no clinical-claim annotations comparable to NatMed.

| Category | Findings | Interpretation |
|---|---|---|
| Structure and completeness | All records are `user_report`; field profiles distinguish missing, null and empty values. Scores and update dates are null on all 4,880 records. | Do not impute popularity, source edits or clinical reliability. Empty comment titles are not missing post titles. |
| Identifiers | Record/native IDs, source/native identity, record type, URL host and timestamps pass implemented checks; passage and graph IDs are unique. | IDs preserve source context; no new identities or merges are created. |
| Duplicates and ambiguity | 24 exact repeated record-text groups covering 230 records; 25 repeated passage-text groups; no identical record payloads after excluding ID/native ID. | Repeated text in distinct threads is not automatically a duplicate source record. Passage overlap is not independent evidence. |
| Mapping quality | 1,285 `own_text` records and 3,595 `thread_context` records; own-text records have at least one literal captured alias in title/text. | This is a lexical check, not verification of every mapped topic, use, benefit or adverse effect. Inherited topics are not direct mentions. |
| Provenance | Manifest/text hashes, all 5,071 passage word windows and metadata bindings, graph endpoints and source-topic agreement pass. | Traceability does not certify factual accuracy, authorship, representativeness or current source availability. |

### Context findings

- 60 comments lack a saved thread post.
- 53 comments have a recorded parent ID that is outside the saved sample.
- Another 99 comments have **no parent ID recorded**. Thus 152 lack a resolvable parent, broader than the earlier audit's 53 external-parent references.
- 159 comments have at least one thread/parent context gap; categories overlap.
- The existing graph has 4,890 nodes and 7,982 `MENTIONS` edges, with no repeated directed typed edges. It does not distinguish direct versus inherited topic meaning in the edge type; retain `mention_scope` and source provenance when interpreting these links.
- The sample includes r/Supplements and r/herbalism; saved publication dates span 2020–2026. This is a bounded sample, not a complete or representative archive.

### What “schema and source support” means here

No dedicated Reddit clinical-claim JSON Schema is supplied in the imported artifacts. **NatMed's clinical-claim schema is not applied to Reddit.** This audit uses explicit local structural checks for source records, IDs/types/roles/URLs/timestamps, source hashes, passage offsets and inherited context, and graph/source-topic consistency. It does not claim formal upstream JSON Schema certification.

No indication, dose, population, effectiveness rating or interaction statement is extracted or clinically approved. The saved records remain user reports. If structured experience annotations are later added, they need a separately defined schema and evidence rules.

The following cells show field completeness and actual source examples. No sources are corrected, merged or re-fetched. Personal information can still appear in text even without usernames; the audit does not de-identify text.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

reddit_root = Path.cwd()
if not (reddit_root / 'data/raw/reddit').is_dir():
    reddit_root = reddit_root.parent
reddit_run = subprocess.run(
    [sys.executable, str(reddit_root / 'scripts/audit_reddit.py')],
    cwd=reddit_root, check=True, capture_output=True, text=True,
)
reddit_audit_dir = Path(reddit_run.stdout.splitlines()[0])
reddit_audit_report = json.loads((reddit_audit_dir / 'audit.json').read_text())
reddit_field_profiles = json.loads((reddit_audit_dir / 'field_profiles.json').read_text())
reddit_records = {r['id']:r for r in map(json.loads,(reddit_root / 'data/raw/reddit/records.jsonl').read_text().splitlines())}
reddit_passages = [json.loads(line) for line in (reddit_root / 'data/raw/reddit/upstream_derived/passages.jsonl').read_text().splitlines()]
print('Report:', reddit_audit_dir.relative_to(reddit_root))
print(json.dumps(reddit_audit_report['summary'], indent=2))
print('RECORD FIELD COMPLETENESS')
print(json.dumps(reddit_field_profiles['record'], indent=2))


Report: data/processed/reddit/reddit-baseline-audit-v1-058aea355a5bb848
{
  "records": 4880,
  "record_types": {
    "comment": 4593,
    "post": 287
  },
  "passages": 5071,
  "evidence_roles": {
    "user_report": 4880
  },
  "mention_scopes": {
    "thread_context": 3595,
    "own_text": 1285
  },
  "duplicate_text_groups": 24,
  "records_in_duplicate_text_groups": 230,
  "duplicate_passage_text_groups": 25,
  "duplicate_record_payload_groups": 0,
  "comments_without_saved_thread": 60,
  "comments_without_saved_parent": 152,
  "comments_with_no_parent_id": 99,
  "comments_with_parent_id_outside_sample": 53,
  "comments_with_either_context_gap": 159,
  "own_text_records_without_literal_alias_hit": 0,
  "deleted_removed_placeholder_records": 0,
  "graph_nodes": 4890,
  "graph_edges": 7982,
  "graph_edge_types": {
    "MENTIONS": 7982
  },
  "duplicate_directed_graph_edges": 0,
  "subreddit_date_coverage": {
    "Supplements": {
      "records": 2426,
      "earliest": "2026-09-21T19:0

In [ ]:
print('EXAMPLE SOURCE RECORDS — USER REPORTS, NOT CLINICAL CLAIMS')
examples = {}
for record in reddit_records.values():
    examples.setdefault((record['record_type'], record['mention_scope']), record)
for key, record in sorted(examples.items()):
    print('\nRecord type / mention scope:', key)
    print(json.dumps(record, ensure_ascii=False, indent=2))
print('CONTEXT-GAP EXAMPLE')
record_id = reddit_audit_report['context_review']['missing_parents'][0]
print(json.dumps(reddit_records[record_id], ensure_ascii=False, indent=2))
print('EXACT-TEXT DUPLICATE EXAMPLE — DIFFERENT SOURCE CONTEXTS RETAINED')
group = reddit_audit_report['duplicates']['record_text'][0]
for record_id in group[:3]:
    print(json.dumps({k:reddit_records[record_id].get(k) for k in ['id','text','thread_id','parent_id','source_url','mention_scope']}, ensure_ascii=False, indent=2))
print('PASSAGE / SOURCE SUPPORT EXAMPLE')
p = reddit_passages[0]
r = reddit_records[p['record_id']]
print(json.dumps({'passage':p, 'reconstructed_text':' '.join(r['text'].split()[p['word_start']:p['word_start']+p['word_count']])}, ensure_ascii=False, indent=2))


EXAMPLE SOURCE RECORDS — USER REPORTS, NOT CLINICAL CLAIMS

Record type / mention scope: ('comment', 'own_text')
{
  "id": "reddit:t1_19rg2tmlzbgg4",
  "native_id": "t1_19rg2tmlzbgg4",
  "source": "reddit",
  "evidence_role": "user_report",
  "record_type": "comment",
  "source_url": "https://old.reddit.com/r/Supplements/comments/1xbk4mh/who_here_is_using_melatonin_nightly_and_did_a/19rg2tmlzbgg4/",
  "collected_at": "2026-10-27T12:49:16.285332+00:00",
  "source_created_at": "2026-10-21T18:25:37+00:00",
  "source_updated_at": null,
  "title": "",
  "text": "Totally disagree…the gland responsible for producing melatonin actually shrivels up and stops producing melatonin for women in menopause…so it’s necessary for women if needed and will be used by me indefinitely 🤩",
  "supplements": [
    {
      "supplement": "Melatonin",
      "matched_terms": [
        "melatonin"
      ]
    }
  ],
  "mention_scope": "own_text",
  "thread_id": "t3_1xbk4mh",
  "parent_id": "t1_pi03qrc",
  "subredd

## 20. Reddit POC field selection — provisional

**Purpose:** support discovery of user questions and clearly attributed personal experiences, with enough context to avoid misrepresenting an inherited topic as a direct report. This is a field selection, not the finalized shared schema or approval to extract clinical facts. `read.md` remains unchanged until shared-schema finalization.

| Category | Selected existing fields | Rule |
|---|---|---|
| Source identity | `id`, `native_id`, `record_type` | Keep posts/comments distinct and retain stable source identities. |
| User-visible content | `title`, `text` | Preserve original wording and qualifications. Comment titles may be empty. Do not silently paraphrase a source into a clinical assertion. |
| Topic candidates | `supplements` including `supplement` and `matched_terms`; `mention_scope`; `normalization` | Keep alias-matched topics provisional. `thread_context` is inherited context, not proof the person used or mentioned the supplement. |
| Conversation context | `thread_id`, `parent_id`, `subreddit` | Keep missing references explicit. Resolve within the captured set where possible; do not invent missing parent text. |
| Evidence role / status | `source`, `evidence_role`, `requires_clinical_review` | Retain `user_report`. This is not medical evidence or a NatMed-style pending clinical claim. No approval state is invented. |
| Provenance and dates | `source_url`, `source_created_at`, `source_updated_at`, `collected_at`, `first_collected_at`, `content_hash`, `schema_version` | Publication and collection dates have different meanings. Keep null update dates unknown. |
| Passage retrieval | `id`, `record_id`, `text`, `title`, `section`, `section_index`, `word_start`, `word_count`, `chunk_words`, `source_content_hash` | Keep exact source binding and overlap information. Count underlying records, not overlapping passages, when reporting source counts. |
| Passage context | `source`, `evidence_role`, `record_type`, `source_url`, `mention_scope`, `thread_id`, `parent_id`, `supplements`, `normalization`, available dates and review flags | Preserve context inherited from the source record; do not promote inherited mentions into direct statements. |
| Derived audit metadata | Missing-thread/parent flags, duplicate-text candidate group, source-validation status, extraction/audit-rule version | These are local audit annotations, not original source facts. Keep traceability to the report and source record. |

**Not selected as structured clinical fields:** indication, population, dose, effectiveness/safety ratings, interaction partner or evidence grade. The existing Reddit dataset has not extracted or validated these. They may occur in text, but adding structured experience annotations is a separate future task, requiring its own schema and source support.

**Citations:** retain the Reddit source URL for attribution. Shared-format passage fields such as `references`, `citation_links`, `reference_scope` and `citation_scope` do not constitute a verified bibliography or claim support. Preserve the original artifacts; do not manufacture study links from them.

**Deferred from the POC projection:** `score` (null throughout this snapshot), NatMed-specific null capture/clinical metadata, upstream graph exports as independent evidence, and any proposed author/profile fields. Do not add usernames. Free text still requires suitable privacy handling before publication or evaluation use.

**Deduplication boundary:** reuse source identities, but do not collapse posts/comments on identical text alone. A later retrieval layer can avoid redundant excerpts while preserving each source/context. Keep direct and inherited-topic associations distinguishable.

**Next:** complete the remaining dataset selections, then finalize the shared schema. No clinical annotations, normalization, corrections or merges are performed by this selection.


## 21. MeSH baseline audit — shared five categories

**Scope:** offline audit of the saved 2026-10-28 MeSH RDF collection. No new queries, mapping approvals or clinical validation.

| Category | Finding | Interpretation |
|---|---|---|
| Structure/completeness | 739 queries, 270 matched, 469 no match; 247 candidate records with labels and identifiers | Records include 195 TopicalDescriptor, 49 SCR_Chemical and 3 SCR_Organism; do not flatten these types |
| Identifiers/source links | All 3,391 DSLD ingredient/form occurrences verified across 400 labels; 270 entry-term URIs resolve to candidate records | Term, concept and descriptor/supplementary-record identifiers have different roles |
| Duplicates/ambiguity | 264 single-record queries; 6 ambiguous queries; 23 records reached by multiple queries | A single candidate does not establish ingredient equivalence |
| RDF duplication | 24,792 unique statements; 400 repeated occurrences; 802 label-text groups shared across distinct RDF subjects | Statement repetition across record subgraphs and shared term/concept labels do not make all those entities duplicates |
| Mapping scope | 206 RDF subjects have scope notes; 52 records have no tree numbers | Scope notes describe terminology; supplementary records need not have descriptor hierarchy placement |
| Provenance/hierarchy | Manifest/source hashes, query occurrences, term resolution and candidate-index checks pass. 791 selected relation edges point to targets without captured descriptions | The saved graph is bounded; external targets are not automatically invalid |

The endpoint was `current`. Available-year metadata stayed unchanged, but that does not prove every response belongs to an immutable pinned MeSH release. Preserve raw request metadata and collection dates.

This audit checks the saved JSON/SPARQL structures and references; it is not full RDF ontology validation, live term-status verification, or clinical review. `broaderDescriptor`, `broaderConcept`, `mappedTo` and `preferredMappedTo` are distinct relations; none should automatically become an ingredient-equivalence or efficacy relation. All mappings remain unreviewed.

Examples below show original lookup responses, term-to-concept-to-record links and selected RDF properties. Query scope includes DSLD co-ingredients and forms, but excludes `otheringredients`. No-match is specific to these saved exact entry-term lookups.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

mesh_root = Path.cwd()
if not (mesh_root / 'data/raw/mesh').is_dir():
    mesh_root = mesh_root.parent
mesh_run = subprocess.run([sys.executable, str(mesh_root / 'scripts/audit_mesh.py')],
                          cwd=mesh_root, check=True, capture_output=True, text=True)
mesh_audit_dir = Path(mesh_run.stdout.splitlines()[0])
mesh_audit_report = json.loads((mesh_audit_dir / 'audit.json').read_text())
mesh_field_profiles = json.loads((mesh_audit_dir / 'field_profiles.json').read_text())
mesh_source = json.loads((mesh_root / 'data/raw/mesh/runs/20261028T062833838481Z/collection.json').read_text())
print('Report:', mesh_audit_dir.relative_to(mesh_root))
print(json.dumps(mesh_audit_report['summary'], indent=2))
print('RECORD PROJECTION COMPLETENESS — EMPTY ARRAYS MEAN NO CAPTURED VALUES')
print(json.dumps(mesh_field_profiles['record_projection'], indent=2))


Report: data/processed/mesh/mesh-baseline-audit-v1-be204bc83020940e
{
  "source_labels": 400,
  "query_groups": 739,
  "source_name_variants": 753,
  "source_occurrences_verified": 3391,
  "search_status": {
    "not_found": 469,
    "matched": 270
  },
  "single_record_queries": 264,
  "ambiguous_queries": 6,
  "candidate_links": 276,
  "candidate_records": 247,
  "record_types": {
    "SCR_Chemical": 49,
    "SCR_Organism": 3,
    "TopicalDescriptor": 195
  },
  "shared_records_across_queries": 23,
  "unique_entry_terms": 270,
  "unique_rdf_subjects": 2360,
  "unique_rdf_statements": 24792,
  "repeated_triple_occurrences": 400,
  "subjects_with_scope_notes": 206,
  "records_missing_labels": 0,
  "records_missing_identifiers": 0,
  "records_without_tree_numbers": 52,
  "shared_label_groups_across_rdf_subjects": 802,
  "relation_counts": {
    "preferredMappedTo": 62,
    "mappedTo": 9,
    "narrowerConcept": 487,
    "relatedConcept": 64,
    "broaderConcept": 11,
    "broaderDescript

In [ ]:
mesh_review_by_query = {r['query_key']:r for r in mesh_audit_report['mapping_review']}
mesh_records_by_uri = {r['uri']:r for r in mesh_audit_report['record_projection']}
for key in ['magnesium','magnesium citrate','creatine','creatine monohydrate','creatine hcl','turmeric','ashwagandha']:
    row = mesh_review_by_query[key]
    print('\nQUERY:', row['query'])
    print(json.dumps({'source_occurrence':mesh_source['queries'][key]['occurrences'][0],
        'raw_lookup_response':mesh_source['search_results'][key]['response'],
        'term_resolution':row['resolution_links'],
        'candidate_records':[mesh_records_by_uri[c['uri']] for c in row['candidates']]}, ensure_ascii=False, indent=2))
print('ALL AMBIGUOUS QUERIES — CANDIDATES ONLY')
for row in mesh_audit_report['mapping_review']:
    if len(row['candidates']) > 1:
        print(json.dumps(row, ensure_ascii=False, indent=2))
print('SHARED RECORD EXAMPLES')
print(json.dumps(mesh_audit_report['shared_records'][:5], ensure_ascii=False, indent=2))
print('RELATION EXAMPLES WITH TARGETS OUTSIDE CAPTURED DESCRIPTIONS')
print(json.dumps([r for r in mesh_audit_report['relations'] if not r['target_description_in_snapshot']][:5], indent=2))



QUERY: Magnesium
{
  "source_occurrence": {
    "dsld_id": "19279",
    "search_categories": [
      "Magnesium"
    ],
    "ingredient_id": 69771,
    "name": "Magnesium",
    "kind": "ingredient",
    "json_path": "ingredientRows[0]"
  },
  "raw_lookup_response": [
    {
      "resource": "http://id.nlm.nih.gov/mesh/T024677",
      "label": "Magnesium"
    }
  ],
  "term_resolution": [
    {
      "term": "http://id.nlm.nih.gov/mesh/T024677",
      "concept": "http://id.nlm.nih.gov/mesh/M0012884",
      "record": "http://id.nlm.nih.gov/mesh/D008274",
      "type": "http://id.nlm.nih.gov/mesh/vocab#TopicalDescriptor"
    }
  ],
  "candidate_records": [
    {
      "uri": "http://id.nlm.nih.gov/mesh/D008274",
      "types": [
        "http://id.nlm.nih.gov/mesh/vocab#TopicalDescriptor"
      ],
      "labels": [
        "Magnesium"
      ],
      "identifiers": [
        "D008274"
      ],
      "preferred_concepts": [
        "http://id.nlm.nih.gov/mesh/M0012884"
      ],
      "conc

## 22. MeSH POC field selection — provisional

**Purpose:** supply controlled terminology, scope descriptions and vocabulary relationships for ingredient/topic mapping and retrieval. Selection does not finalize the shared schema or approve equivalence with DSLD, PubChem, RxNorm or clinical claims.

MeSH data is stored as lookup responses, resolution bindings and RDF subgraphs, not a flat table. The following are source fields or RDF predicates; preserve full namespace URIs and the distinction between record, concept and term subjects.

| Category | Selected existing fields / predicates | Rule |
|---|---|---|
| Record identity | Record URI, `rdf:type`, `meshv:identifier`, `rdfs:label` | Keep descriptors and supplementary chemical/organism records distinct; namespace identifiers. |
| Concept/term identity | Resolution `term`, `concept`, `record`, `type`; `meshv:concept`, `preferredConcept`, `term`, `preferredTerm` | Retain the path from the matched entry term to its concept and record. A term/concept/record sharing a label does not mean duplicate entities. |
| Names and scope | `meshv:prefLabel`, `altLabel`, `scopeNote`; source `rdfs:label` | Keep aliases and descriptions attached to their actual RDF subject. They are not validated clinical assertions or automatic merge keys. |
| Hierarchy | `meshv:treeNumber`, `broaderDescriptor`, `broaderConcept`, `narrowerConcept` | Preserve relation direction/type and multiple placements. Missing tree numbers are not automatically invalid, particularly for supplementary records. |
| Supplementary mapping/context | `meshv:mappedTo`, `preferredMappedTo`, `relatedConcept`, `seeAlso` | Keep these as distinct vocabulary links, not chemical identity or indication relationships. Targets may be outside the sample. |
| Status and dates | Available `meshv:active`, `lastActiveYear`, `dateCreated`, `dateEstablished`, `dateIntroduced`, `dateRevised`, `lastUpdated` | Preserve source datatypes and absent fields. These dates differ from capture time and clinical-study dates. |
| Query identity | `queries` key, `query`, `variants` | Keep the original lookup names, including form-qualified names. |
| DSLD source occurrence | `occurrences`: `dsld_id`, `ingredient_id`, `name`, `kind`, `json_path`, `search_categories` | Preserve source ingredient/form context. Retrieval category is not an indication or identity approval. |
| Mapping result | `search_results`: `status`, `response`, `term_uris`, `possibly_truncated`, `match_method`, `requires_identity_review`; `candidate_record_index` | Preserve complete candidate sets, ambiguity, no-match and truncation status. A no-match review flag does not imply approved identity. |
| Source support | Lookup request URL/params/timestamps/status/response; term-resolution and record-subgraph batches including `complete`, `pages`, `requested_terms` or `requested_records` | Link each selected value/relation to its raw response context. Repeated statements may share a graph fact while retaining source occurrences later. |
| Dataset provenance | `source`, `scope`, `search_policy`, collection timestamps, source metadata at start/end, `dsld_provenance`, source-file hashes | `current` plus unchanged available-year metadata is not a verified immutable release. |
| Review metadata (derived) | Candidate status, unresolved-target flags, validation result, audit/normalization rule version | Keep returned candidates unreviewed. No confidence score or clinical approval is inferred. |

**RDF value preservation:** retain object `type`, `value`, `datatype` and language tag when present. A literal is not a URI, and strings with different datatypes/languages must not be collapsed merely because their displayed text matches.

**Clinical-field boundary:** this collection does not provide reviewed clinical claim types, population-specific indications, effectiveness/safety ratings, dose regimens, interaction partners or study evidence. Mark these as not supplied by this dataset; do not infer them from labels, scope notes, hierarchy or pharmacological-action indexing. MeSH source URIs and RDF statements provide terminology provenance, not citations establishing clinical efficacy.

**Deferred from the initial graph projection:** allowable qualifiers, indexing instructions, historical/public notes, classification codes, registry-number mappings and pharmacological-action links unless a later specific use case requires them. Retain them in raw RDF. Do not perform an unrestricted crawl of external hierarchy targets or manufacture their missing descriptions.

**Examples:** Magnesium resolves to descriptor D008274, while Magnesium Citrate resolves to supplementary record C110422. Creatine resolves to D003401; its monohydrate and HCl queries return no match here. Turmeric and Ashwagandha have candidates even though those literal PubChem name searches returned none. Different coverage across vocabularies does not prove or disprove ingredient identity.

**Selection only:** retain unknowns, source distinctions and candidate status. No records are merged, mappings approved or clinical claims created. Finalize the shared schema after reviewing all dataset selections together. `read.md` remains unchanged.
